In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

In [ ]:
!pip install upsetplot

In [ ]:
registry = pd.read_csv("../Data/Raw/approved_slice_registry_FINAL.csv")
global_results = pd.read_csv("nb6_global_results.csv")
worst_slice_results = pd.read_csv("nb6_worst_slice_results.csv")
delta_combined = pd.read_csv("nb6_delta_vs_raw__combined_repair.csv")

print("Registry:", registry.shape)
print("Global:", global_results.shape)
print("Worst-slice:", worst_slice_results.shape)
print("Delta combined:", delta_combined.shape)

In [ ]:
# ============================================================
# FIGURE 1
# Conceptual Illustration of Hidden Failure Slices
# IEEE High-Resolution Version
# ============================================================

import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Ellipse
import matplotlib.patches as patches

plt.close("all")

fig, ax = plt.subplots(figsize=(13,7))

ax.set_xlim(0,100)
ax.set_ylim(0,70)
ax.axis("off")

# ======================================================
# ED Population
# ======================================================

population = Circle(
    (18,35),
    14,
    facecolor="#DCEEFF",
    edgecolor="navy",
    linewidth=2.5
)

ax.add_patch(population)

ax.text(
    18,
    53,
    "Entire ED Population",
    ha="center",
    fontsize=14,
    fontweight="bold"
)

# ======================================================
# Hidden Slice
# ======================================================

slice_region = Ellipse(
    (22,30),
    width=10,
    height=7,
    angle=-20,
    facecolor="#F4A261",
    edgecolor="darkred",
    linewidth=2
)

ax.add_patch(slice_region)

ax.text(
    23,
    20,
    "Hidden Failure Slice",
    ha="center",
    fontsize=12,
    fontweight="bold",
    color="darkred"
)

# ======================================================
# Aggregate Evaluation
# ======================================================

aggregate_box = patches.FancyBboxPatch(
    (42,43),
    28,
    16,
    boxstyle="round,pad=0.5",
    linewidth=2,
    edgecolor="#2E8B57",
    facecolor="#EAF7EA"
)

ax.add_patch(aggregate_box)

ax.text(
    56,
    56,
    "Aggregate Evaluation",
    fontsize=14,
    fontweight="bold",
    ha="center"
)

ax.text(
    56,
    49,
    "• Population-level assessment",
    fontsize=11,
    ha="center"
)

ax.text(
    56,
    45,
    "• Satisfactory overall performance",
    fontsize=11,
    ha="center"
)

# ======================================================
# Slice Evaluation
# ======================================================

slice_box = patches.FancyBboxPatch(
    (42,10),
    38,
    22,
    boxstyle="round,pad=0.5",
    linewidth=2,
    edgecolor="#B22222",
    facecolor="#FFF2F2"
)

ax.add_patch(slice_box)

ax.text(
    61,
    28,
    "Slice-Level Evaluation",
    fontsize=14,
    fontweight="bold",
    ha="center"
)

ax.text(
    61,
    23,
    "• Elevated prediction loss",
    fontsize=11,
    ha="center"
)

ax.text(
    61,
    19,
    "• Reduced recall",
    fontsize=11,
    ha="center"
)

ax.text(
    61,
    15,
    "• Calibration deviation",
    fontsize=11,
    ha="center"
)

# ======================================================
# Arrows
# ======================================================

ax.annotate(
    "",
    xy=(42,50),
    xytext=(31,43),
    arrowprops=dict(
        arrowstyle="-|>",
        lw=2.5,
        color="#2E8B57"
    )
)

ax.annotate(
    "",
    xy=(42,22),
    xytext=(27,30),
    arrowprops=dict(
        arrowstyle="-|>",
        lw=2.5,
        color="#B22222"
    )
)

# ======================================================
# Note
# ======================================================

note = patches.FancyBboxPatch(
    (78,22),
    20,
    18,
    boxstyle="round,pad=0.5",
    linewidth=2,
    edgecolor="gray",
    facecolor="#F8F8F8"
)

ax.add_patch(note)

ax.text(
    88,
    35,
    "Key Insight",
    fontsize=13,
    fontweight="bold",
    ha="center"
)

ax.text(
    88,
    30,
    "Subgroup defined by",
    fontsize=10,
    ha="center"
)

ax.text(
    88,
    27,
    "clinical and operational",
    fontsize=10,
    ha="center"
)

ax.text(
    88,
    24,
    "variables only",
    fontsize=10,
    ha="center"
)

ax.text(
    88,
    20,
    "(not protected attributes)",
    fontsize=9,
    ha="center",
    color="darkred"
)

# ======================================================
# Overall Title
# ======================================================

plt.title(
    "Conceptual Illustration of Hidden Failure Slices in Emergency Department AI",
    fontsize=16,
    fontweight="bold",
    pad=20
)

plt.tight_layout()

plt.savefig(
    "Figure1_Conceptual_Hidden_Failure_Slice.pdf",
    dpi=600,
    bbox_inches="tight"
)

plt.savefig(
    "Figure1_Conceptual_Hidden_Failure_Slice.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()

# Figure 5 — Global Predictive Performance versus Hidden Subgroup Evaluation

In [ ]:
# ============================================================
# FIGURE 3 — GLOBAL PERFORMANCE VS HIDDEN SUBGROUP HETEROGENEITY
# FINAL IEEE PUBLICATION VERSION
#
# Panel (a): Aggregate ROC on OOF diagnostic cohort
# Panel (b): Aggregate Precision–Recall on OOF diagnostic cohort
# Panel (c): ALL positive-harm BASiC slices vs global AUPRC
#
# Key methodological rule:
#   Figure 3 = predictive heterogeneity
#   Figure 4 = operational harm decomposition/ranking
#
# Panel C:
#   - includes all BASiC slices with H_k > 0
#   - plots slice AUPRC on one unambiguous axis
#   - reports Recall and H_k in a separate value column
#   - excludes monitoring-only/non-positive-harm slices
#   - does NOT encode ASAP governance or repair priority
#
# Outputs:
#   PDF (vector)
#   SVG (vector)
#   PNG (600 dpi)
# ============================================================

import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_curve,
    precision_recall_curve,
    roc_auc_score,
    average_precision_score
)

warnings.filterwarnings("ignore")
plt.close("all")

# ============================================================
# 0. OUTPUT DIRECTORY
# ============================================================

FIG_DIR = Path.cwd() / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

PDF_PATH = FIG_DIR / "Fig3_Global_vs_Hidden_Subgroup_Heterogeneity.pdf"
SVG_PATH = FIG_DIR / "Fig3_Global_vs_Hidden_Subgroup_Heterogeneity.svg"
PNG_PATH = FIG_DIR / "Fig3_Global_vs_Hidden_Subgroup_Heterogeneity.png"

# ============================================================
# 1. IEEE-FRIENDLY TYPOGRAPHY
# ============================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7.2,
    "ytick.labelsize": 7.2,
    "legend.fontsize": 7,
    "axes.linewidth": 0.8,
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# ============================================================
# 2. LOAD OOF DIAGNOSTIC DATA
# ============================================================

diagnostic_candidates = [
    "df_with_diagnostics.csv",
    "triage_train_diagnostics.csv"
]

diagnostic_file = next(
    (f for f in diagnostic_candidates if Path(f).exists()),
    None
)

if diagnostic_file is None:
    raise FileNotFoundError(
        "Could not find an OOF diagnostic file.\n"
        "Expected one of:\n"
        + "\n".join(diagnostic_candidates)
    )

diag = pd.read_csv(diagnostic_file)

print("=" * 76)
print("FIGURE 3 — DATA SOURCES")
print("=" * 76)

print("Diagnostic source:", diagnostic_file)
print("Diagnostic shape :", diag.shape)

# ============================================================
# 3. IDENTIFY TARGET COLUMN
# ============================================================

target_candidates = [
    "y",
    "admitted",
    "admission",
    "admission_label",
    "target",
    "label",
    "outcome"
]

target_col = next(
    (c for c in target_candidates if c in diag.columns),
    None
)

if target_col is None:
    raise ValueError(
        "Could not identify the binary outcome column.\n"
        f"Available columns:\n{list(diag.columns)}"
    )

# ============================================================
# 4. IDENTIFY OOF PROBABILITY COLUMN
# ============================================================

prob_candidates = [
    "oof_pred_proba",
    "oof_probability",
    "oof_prob",
    "oof_pred_prob",
    "oof_prediction",
    "predicted_probability",
    "prediction_probability",
    "p_oof",
    "p_hat",
    "y_prob",
    "probability"
]

prob_col = next(
    (c for c in prob_candidates if c in diag.columns),
    None
)

if prob_col is None:

    possible = [
        c for c in diag.columns
        if pd.api.types.is_numeric_dtype(diag[c])
        and (
            "oof" in c.lower()
            or "prob" in c.lower()
        )
    ]

    if possible:
        prob_col = possible[0]

if prob_col is None:
    raise ValueError(
        "Could not identify the OOF probability column.\n"
        f"Available columns:\n{list(diag.columns)}"
    )

print("Outcome column   :", target_col)
print("OOF probability :", prob_col)

# ============================================================
# 5. CLEAN GLOBAL OOF DATA
# ============================================================

global_df = diag[
    [target_col, prob_col]
].copy()

global_df[target_col] = pd.to_numeric(
    global_df[target_col],
    errors="coerce"
)

global_df[prob_col] = pd.to_numeric(
    global_df[prob_col],
    errors="coerce"
)

global_df = global_df.dropna()

global_df = global_df[
    global_df[target_col].isin([0, 1])
].copy()

y_true = global_df[target_col].astype(int).to_numpy()

y_prob = np.clip(
    global_df[prob_col].astype(float).to_numpy(),
    0,
    1
)

if len(np.unique(y_true)) != 2:
    raise ValueError(
        "The target must contain both classes."
    )

# ============================================================
# 6. GLOBAL ROC / PR METRICS
# ============================================================

fpr, tpr, _ = roc_curve(
    y_true,
    y_prob
)

precision, recall_curve, _ = precision_recall_curve(
    y_true,
    y_prob
)

global_auroc = roc_auc_score(
    y_true,
    y_prob
)

global_auprc = average_precision_score(
    y_true,
    y_prob
)

prevalence = float(
    y_true.mean()
)

print(f"OOF observations   : {len(y_true):,}")
print(f"Global AUROC       : {global_auroc:.4f}")
print(f"Global AUPRC       : {global_auprc:.4f}")
print(f"Outcome prevalence : {prevalence:.4f}")

# ============================================================
# 7. HELPER FUNCTIONS
# ============================================================

def find_col(df, exact_candidates, keyword_groups=None):

    for c in exact_candidates:
        if c in df.columns:
            return c

    if keyword_groups:

        for keywords in keyword_groups:

            for c in df.columns:

                cl = str(c).lower()

                if all(
                    keyword.lower() in cl
                    for keyword in keywords
                ):
                    return c

    return None


def canonical_slice_name(name):

    text = str(name).strip().lower()

    text = re.sub(
        r"[\s\-/]+",
        "_",
        text
    )

    text = re.sub(
        r"_+",
        "_",
        text
    )

    return text.strip("_")


def pretty_slice_name(name):

    text = canonical_slice_name(name)

    text = text.replace(
        "_and_",
        " + "
    )

    text = text.replace(
        "_",
        " "
    )

    replacements = {
        "hr missing": "HR missing",
        "spo2 missing": "SpO₂ missing",
        "ckd": "CKD",
        "chd": "CHD",
        "dm": "DM",
        "htn": "HTN",
        "vitals missing ge 3": "vitals missing ≥ 3"
    }

    for old, new in replacements.items():

        text = re.sub(
            rf"\b{re.escape(old)}\b",
            new,
            text,
            flags=re.IGNORECASE
        )

    if text:
        text = text[0].upper() + text[1:]

    return text

# ============================================================
# 8. LOAD SLICE PERFORMANCE DATA
# ============================================================

performance_candidates = [
    "Fig5_slice_level_data.csv",
    "nb8_repair_eligible_slice_results.csv",
    "nb8_top_ranked_repair_slices.csv",
    "nb8_basic_slice_rank_table.csv"
]

performance_file = next(
    (f for f in performance_candidates if Path(f).exists()),
    None
)

if performance_file is None:
    raise FileNotFoundError(
        "Could not find slice-level performance data.\n"
        "Expected one of:\n"
        + "\n".join(performance_candidates)
    )

perf = pd.read_csv(
    performance_file
)

print("\nSlice-performance source:", performance_file)
print("Performance shape        :", perf.shape)

slice_name_col = find_col(
    perf,
    [
        "slice_name",
        "slice",
        "slice_id",
        "name"
    ]
)

slice_auprc_col = find_col(
    perf,
    [
        "AUPRC",
        "auprc",
        "slice_auprc",
        "val_auprc"
    ],
    [
        ["auprc"]
    ]
)

slice_recall_col = find_col(
    perf,
    [
        "Recall",
        "recall",
        "slice_recall",
        "val_recall"
    ],
    [
        ["recall"]
    ]
)

if (
    slice_name_col is None
    or slice_auprc_col is None
    or slice_recall_col is None
):
    raise ValueError(
        "Slice performance file must contain slice name, "
        "AUPRC, and Recall.\n\n"
        f"Columns:\n{list(perf.columns)}"
    )

perf_table = perf[
    [
        slice_name_col,
        slice_auprc_col,
        slice_recall_col
    ]
].copy()

perf_table.columns = [
    "slice_name",
    "auprc",
    "recall"
]

perf_table["slice_key"] = (
    perf_table["slice_name"]
    .apply(canonical_slice_name)
)

perf_table["auprc"] = pd.to_numeric(
    perf_table["auprc"],
    errors="coerce"
)

perf_table["recall"] = pd.to_numeric(
    perf_table["recall"],
    errors="coerce"
)

perf_table = (
    perf_table
    .dropna(
        subset=[
            "slice_key",
            "auprc",
            "recall"
        ]
    )
    .drop_duplicates(
        subset="slice_key",
        keep="first"
    )
)

# ============================================================
# 9. LOAD BASiC HARM DATA
# ============================================================

harm_candidates = [
    "nb8_slice_source_audit.csv",
    "nb8_asap_complete_candidate_audit.csv",
    "nb8_asap_candidate_pool_with_ablations.csv",
    "nb8_composite_harm_components.csv",
    "slice_harm.csv"
]

harm_file = next(
    (f for f in harm_candidates if Path(f).exists()),
    None
)

if harm_file is None:
    raise FileNotFoundError(
        "Could not find a BASiC harm/provenance file."
    )

harm_raw = pd.read_csv(
    harm_file
)

print("Harm/provenance source    :", harm_file)
print("Harm data shape           :", harm_raw.shape)

# ============================================================
# 10. IDENTIFY HARM COLUMNS
# ============================================================

harm_slice_col = find_col(
    harm_raw,
    [
        "slice_name",
        "slice",
        "slice_id",
        "name"
    ]
)

harm_value_col = find_col(
    harm_raw,
    [
        "nb8_harm",
        "H_k",
        "h_k",
        "slice_harm",
        "harm_score",
        "harm"
    ],
    [
        ["harm"]
    ]
)

if harm_slice_col is None:

    unnamed = [
        c for c in harm_raw.columns
        if str(c).lower().startswith("unnamed")
    ]

    if unnamed:
        harm_slice_col = unnamed[0]

if harm_value_col is None:

    numeric_candidates = [
        c for c in harm_raw.columns
        if c != harm_slice_col
        and pd.api.types.is_numeric_dtype(
            harm_raw[c]
        )
    ]

    if len(numeric_candidates) == 1:
        harm_value_col = numeric_candidates[0]

if harm_slice_col is None or harm_value_col is None:

    raise ValueError(
        "Could not identify slice name / operational harm.\n\n"
        f"Available columns:\n{list(harm_raw.columns)}"
    )

harm_table = harm_raw[
    [
        harm_slice_col,
        harm_value_col
    ]
].copy()

harm_table.columns = [
    "slice_name_harm",
    "H_k"
]

harm_table["slice_key"] = (
    harm_table["slice_name_harm"]
    .apply(canonical_slice_name)
)

harm_table["H_k"] = pd.to_numeric(
    harm_table["H_k"],
    errors="coerce"
)

harm_table = (
    harm_table
    .dropna(
        subset=[
            "slice_key",
            "H_k"
        ]
    )
    .drop_duplicates(
        subset="slice_key",
        keep="first"
    )
)

# ============================================================
# 11. RETAIN ALL POSITIVE-HARM BASiC SLICES
# ============================================================

harm_positive = harm_table[
    harm_table["H_k"] > 0
].copy()

print(
    "\nPositive-harm BASiC slices identified:",
    len(harm_positive)
)

# ============================================================
# 12. MERGE HARM WITH PERFORMANCE
# ============================================================

panel_c = harm_positive.merge(
    perf_table[
        [
            "slice_key",
            "slice_name",
            "auprc",
            "recall"
        ]
    ],
    on="slice_key",
    how="left",
    validate="one_to_one"
)

missing_performance = panel_c[
    panel_c[
        ["auprc", "recall"]
    ].isna().any(axis=1)
].copy()

if not missing_performance.empty:

    print(
        "\nWARNING — positive-harm slices without "
        "AUPRC/Recall in the selected performance file:"
    )

    display(
        missing_performance[
            [
                "slice_name_harm",
                "H_k"
            ]
        ]
    )

    panel_c = panel_c.dropna(
        subset=[
            "auprc",
            "recall"
        ]
    )

# ============================================================
# 13. KEEP ALL H_k > 0 SLICES
# ============================================================

panel_c["auprc_gap"] = (
    global_auprc
    - panel_c["auprc"]
)

panel_c["display_name"] = (
    panel_c["slice_key"]
    .apply(pretty_slice_name)
)

panel_c = panel_c.sort_values(
    ["auprc", "H_k"],
    ascending=[
        True,
        False
    ]
).reset_index(drop=True)

if panel_c.empty:
    raise ValueError(
        "No positive-harm BASiC slices have usable "
        "AUPRC/Recall values."
    )

print(
    "Positive-harm slices with performance data:",
    len(panel_c)
)

# ============================================================
# 14. ADAPT FIGURE HEIGHT TO NUMBER OF SLICES
# ============================================================

n_slices = len(panel_c)

panel_c_height = max(
    2.8,
    0.35 * n_slices
)

figure_height = (
    3.3
    + panel_c_height
)

fig = plt.figure(
    figsize=(
        7.16,
        figure_height
    )
)

gs = fig.add_gridspec(
    2,
    2,
    height_ratios=[
        2.15,
        panel_c_height
    ],
    hspace=0.58,
    wspace=0.36
)

ax1 = fig.add_subplot(
    gs[0, 0]
)

ax2 = fig.add_subplot(
    gs[0, 1]
)

ax3 = fig.add_subplot(
    gs[1, :]
)

# ============================================================
# PANEL A — ROC
# ============================================================

ax1.plot(
    fpr,
    tpr,
    linewidth=2.0,
    color="#0072B2",
    label=f"AUROC = {global_auroc:.3f}"
)

ax1.plot(
    [0, 1],
    [0, 1],
    "--",
    linewidth=1.0,
    color="0.55",
    label="Chance"
)

ax1.set_xlabel(
    "False-positive rate"
)

ax1.set_ylabel(
    "True-positive rate"
)

ax1.set_title(
    "(a) Aggregate ROC",
    loc="left",
    fontweight="bold"
)

ax1.set_xlim(
    0,
    1
)

ax1.set_ylim(
    0,
    1.02
)

ax1.grid(
    alpha=0.20,
    linewidth=0.5
)

ax1.legend(
    frameon=False,
    loc="lower right"
)

ax1.spines[
    ["top", "right"]
].set_visible(False)

# ============================================================
# PANEL B — PRECISION–RECALL
# ============================================================

ax2.plot(
    recall_curve,
    precision,
    linewidth=2.0,
    color="#009E73",
    label=f"AUPRC = {global_auprc:.3f}"
)

ax2.axhline(
    prevalence,
    linestyle="--",
    linewidth=1.0,
    color="0.55",
    label=f"Prevalence = {prevalence:.3f}"
)

ax2.set_xlabel(
    "Recall"
)

ax2.set_ylabel(
    "Precision"
)

ax2.set_title(
    "(b) Aggregate precision–recall",
    loc="left",
    fontweight="bold"
)

ax2.set_xlim(
    0,
    1
)

ax2.set_ylim(
    0,
    1.02
)

ax2.grid(
    alpha=0.20,
    linewidth=0.5
)

ax2.legend(
    frameon=False,
    loc="lower left"
)

ax2.spines[
    ["top", "right"]
].set_visible(False)

# ============================================================
# PANEL C — ALL POSITIVE-HARM BASiC SLICES
# ============================================================

rep = (
    panel_c
    .sort_values(
        "auprc",
        ascending=False
    )
    .reset_index(drop=True)
)

y_pos = np.arange(
    len(rep)
)

# ------------------------------------------------------------
# GLOBAL AUPRC REFERENCE
# ------------------------------------------------------------

ax3.axvline(
    global_auprc,
    color="black",
    linestyle="--",
    linewidth=1.4,
    zorder=2
)

# ------------------------------------------------------------
# COLORS INDICATE AUPRC RELATIVE TO GLOBAL ONLY
# ------------------------------------------------------------

COLOR_BELOW = "#D55E00"
COLOR_AT_OR_ABOVE = "#0072B2"

point_colors = np.where(
    rep["auprc"] < global_auprc,
    COLOR_BELOW,
    COLOR_AT_OR_ABOVE
)

# ------------------------------------------------------------
# LOLLIPOP SEGMENTS
# ------------------------------------------------------------

for i, row in rep.iterrows():

    ax3.hlines(
        y=i,
        xmin=min(
            row["auprc"],
            global_auprc
        ),
        xmax=max(
            row["auprc"],
            global_auprc
        ),
        color="0.67",
        linewidth=1.35,
        zorder=1
    )

# ------------------------------------------------------------
# AUPRC POINTS
# ------------------------------------------------------------

ax3.scatter(
    rep["auprc"],
    y_pos,
    s=52,
    c=point_colors,
    edgecolor="black",
    linewidth=0.5,
    zorder=4
)

# ------------------------------------------------------------
# SLICE LABELS
# ------------------------------------------------------------

ax3.set_yticks(
    y_pos
)

ax3.set_yticklabels(
    rep["display_name"],
    fontsize=7.1
)

ax3.invert_yaxis()

# ============================================================
# 15. DYNAMIC PANEL-C X RANGE
# ============================================================

data_min = min(
    float(rep["auprc"].min()),
    global_auprc
)

data_max = max(
    float(rep["auprc"].max()),
    global_auprc
)

data_span = max(
    0.10,
    data_max - data_min
)

x_left = max(
    0,
    data_min - 0.08 * data_span
)

x_right = min(
    1.0,
    data_max + 0.65 * data_span
)

if (
    x_right - data_max
    < 0.12 * data_span
):

    x_right = min(
        1.0,
        data_max + 0.15
    )

ax3.set_xlim(
    x_left,
    x_right
)

# ============================================================
# 16. RIGHT-SIDE NUMERIC COLUMN
# ============================================================

for i, row in rep.iterrows():

    ax3.text(
        1.012,
        i,
        (
            f"AUPRC {row['auprc']:.3f}   "
            f"Recall {row['recall']:.3f}   "
            f"Hₖ {row['H_k']:.3f}"
        ),
        transform=ax3.get_yaxis_transform(),
        va="center",
        ha="left",
        fontsize=6.6,
        clip_on=False
    )

# ------------------------------------------------------------
# METRIC COLUMN HEADER
# ------------------------------------------------------------

ax3.text(
    1.012,
    1.005,
    "Slice-level metrics",
    transform=ax3.transAxes,
    ha="left",
    va="bottom",
    fontsize=6.8,
    fontweight="bold",
    clip_on=False
)

# ============================================================
# 17. GLOBAL REFERENCE LABEL
# ============================================================

ax3.annotate(
    f"Global reference\nAUPRC = {global_auprc:.3f}",
    xy=(
        global_auprc,
        1.0
    ),
    xycoords=(
        "data",
        "axes fraction"
    ),
    xytext=(
        0,
        4
    ),
    textcoords="offset points",
    ha="center",
    va="bottom",
    fontsize=6.8,
    fontweight="bold",
    linespacing=1.15,
    annotation_clip=False
)

# ============================================================
# 18. PANEL C FORMATTING
# ============================================================

ax3.set_xlabel(
    "AUPRC"
)

ax3.set_title(
    "(c) Predictive heterogeneity across positive-harm BASiC slices",
    loc="left",
    fontweight="bold",
    fontsize=9,
    pad=18
)

ax3.grid(
    axis="x",
    alpha=0.20,
    linewidth=0.5
)

ax3.set_axisbelow(True)

ax3.spines[
    ["top", "right"]
].set_visible(False)

# ------------------------------------------------------------
# LEGEND
# ------------------------------------------------------------

below_handle = plt.Line2D(
    [0],
    [0],
    marker="o",
    linestyle="",
    markerfacecolor=COLOR_BELOW,
    markeredgecolor="black",
    markersize=5,
    label="AUPRC < global reference"
)

above_handle = plt.Line2D(
    [0],
    [0],
    marker="o",
    linestyle="",
    markerfacecolor=COLOR_AT_OR_ABOVE,
    markeredgecolor="black",
    markersize=5,
    label="AUPRC ≥ global reference"
)

ax3.legend(
    handles=[
        below_handle,
        above_handle
    ],
    loc="upper center",
    bbox_to_anchor=(
        0.50,
        -0.135
    ),
    frameon=False,
    ncol=2,
    columnspacing=2.0,
    handletextpad=0.6,
    borderaxespad=0.0,
    fontsize=6.8
)

# ============================================================
# 19. FIGURE-LEVEL MESSAGE
# ============================================================

fig.text(
    0.5,
    0.018,
    (
        "Aggregate OOF performance provides a population-level reference; "
        "positive-harm slices reveal heterogeneous local predictive behaviour."
    ),
    ha="center",
    va="bottom",
    fontsize=7.0
)

# ============================================================
# 20. FINAL LAYOUT
# ============================================================

fig.subplots_adjust(
    left=0.285,
    right=0.72,
    top=0.955,
    bottom=0.155,
    hspace=0.58,
    wspace=0.36
)

# ============================================================
# 21. EXPORT — SAVE BEFORE SHOW
# ============================================================

fig.savefig(
    PDF_PATH,
    format="pdf",
    bbox_inches="tight",
    pad_inches=0.08
)

fig.savefig(
    SVG_PATH,
    format="svg",
    bbox_inches="tight",
    pad_inches=0.08
)

fig.savefig(
    PNG_PATH,
    format="png",
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.08
)

# ============================================================
# 22. VERIFY EXPORTS
# ============================================================

print("\n" + "=" * 76)
print("FIGURE 3 EXPORT VERIFICATION")
print("=" * 76)

for label, path in {
    "PDF": PDF_PATH,
    "SVG": SVG_PATH,
    "PNG": PNG_PATH
}.items():

    if not path.exists():
        raise IOError(
            f"{label} was not created: {path}"
        )

    file_size_kb = (
        path.stat().st_size
        / 1024
    )

    if file_size_kb <= 0:
        raise IOError(
            f"{label} appears empty: {path}"
        )

    print(
        f"{label}: {path.resolve()}"
    )

    print(
        f"     size = {file_size_kb:.1f} KB"
    )

plt.show()

plt.close(fig)

# ============================================================
# 23. EXPORT EXACT PANEL-C SOURCE DATA
# ============================================================

panel_c_export = rep[
    [
        "slice_key",
        "auprc",
        "recall",
        "H_k",
        "auprc_gap"
    ]
].copy()

panel_c_export.columns = [
    "slice_name",
    "slice_AUPRC",
    "slice_recall",
    "operational_harm_Hk",
    "global_minus_slice_AUPRC"
]

panel_c_export.to_csv(
    FIG_DIR / "Fig3_PanelC_Positive_Harm_Slices.csv",
    index=False
)

# ============================================================
# 24. FINAL REPORT
# ============================================================

print("\n" + "=" * 76)
print("FIGURE 3 GENERATED SUCCESSFULLY")
print("=" * 76)

print(f"Global AUROC : {global_auroc:.4f}")
print(f"Global AUPRC : {global_auprc:.4f}")
print(f"Prevalence   : {prevalence:.4f}")

print(
    "\nPositive-harm BASiC slices displayed:",
    len(rep)
)

print("\nPanel C source data:")

display(
    panel_c_export.style.format({
        "slice_AUPRC": "{:.4f}",
        "slice_recall": "{:.4f}",
        "operational_harm_Hk": "{:.4f}",
        "global_minus_slice_AUPRC": "{:.4f}"
    })
)

print("\nFigure directory:")
print(FIG_DIR.resolve())

print("\nInterpretation:")
print(
    "All displayed slices satisfy H_k > 0. "
    "Color indicates AUPRC relative to the global reference only."
)

print("=" * 76)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_curve,
    precision_recall_curve,
    roc_auc_score,
    average_precision_score,
    recall_score
)

# Load the locked-test predictions produced by Notebook 6
test_fig_df = pd.read_csv(
    "nb6_test_predictions_with_slices.csv"
)

# Load the finalized slice registry
registry = pd.read_csv(
    "approved_slice_registry_FINAL.csv"
)

accepted_registry = registry[
    registry["final_accept"]
    .astype(str)
    .str.strip()
    .str.lower()
    .isin(["yes", "true", "1"])
].copy()

print("Test evaluation rows:", len(test_fig_df))
print("Accepted slices:", len(accepted_registry))

In [ ]:
MODEL_ARM = "raw_erm"
PROB_COL = f"pred__{MODEL_ARM}"

if PROB_COL not in test_fig_df.columns:
    raise KeyError(
        f"{PROB_COL} was not found. "
        "Check that Notebook 6 exported the raw ERM predictions."
    )

y_true = test_fig_df["y"].astype(int).to_numpy()
y_prob = test_fig_df[PROB_COL].astype(float).to_numpy()

In [ ]:
# Global threshold-independent metrics
global_auroc = roc_auc_score(y_true, y_prob)
global_auprc = average_precision_score(y_true, y_prob)

# Curves
fpr, tpr, roc_thresholds = roc_curve(y_true, y_prob)
precision, recall, pr_thresholds = precision_recall_curve(
    y_true,
    y_prob
)

# Positive-class prevalence is the no-skill PR reference
positive_prevalence = y_true.mean()

print(f"Global AUROC: {global_auroc:.4f}")
print(f"Global AUPRC: {global_auprc:.4f}")
print(f"Positive prevalence: {positive_prevalence:.4f}")

In [ ]:
def calculate_slice_performance(
    data,
    accepted_slice_names,
    probability_column,
    threshold=0.5,
    minimum_slice_size=50
):
    """
    Calculate test-set performance for each accepted slice.

    Parameters
    ----------
    data : pandas.DataFrame
        Must contain y, probability predictions, and slice membership columns.
    accepted_slice_names : list[str]
        Names of accepted slices.
    probability_column : str
        Prediction-probability column.
    threshold : float
        Threshold used to calculate recall.
    minimum_slice_size : int
        Minimum slice size required for reporting.

    Returns
    -------
    pandas.DataFrame
    """
    rows = []

    for slice_name in accepted_slice_names:
        membership_col = f"SLICE__{slice_name}"

        if membership_col not in data.columns:
            print(
                f"Warning: {membership_col} is missing and will be skipped."
            )
            continue

        mask = data[membership_col].astype(int) == 1
        slice_n = int(mask.sum())

        if slice_n < minimum_slice_size:
            continue

        y_slice = data.loc[mask, "y"].astype(int).to_numpy()
        p_slice = data.loc[
            mask,
            probability_column
        ].astype(float).to_numpy()

        # AUPRC requires at least one positive case
        if y_slice.sum() == 0:
            slice_auprc = np.nan
        else:
            slice_auprc = average_precision_score(
                y_slice,
                p_slice
            )

        predicted_slice = (p_slice >= threshold).astype(int)

        slice_recall = recall_score(
            y_slice,
            predicted_slice,
            zero_division=0
        )

        rows.append({
            "slice_name": slice_name,
            "n": slice_n,
            "support_pct": 100 * slice_n / len(data),
            "event_rate": y_slice.mean(),
            "AUPRC": slice_auprc,
            "Recall": slice_recall
        })

    results = pd.DataFrame(rows)

    if results.empty:
        raise ValueError(
            "No eligible accepted slices were found."
        )

    return results


accepted_slice_names = accepted_registry[
    "slice_name"
].tolist()

figure5_slice_results = calculate_slice_performance(
    data=test_fig_df,
    accepted_slice_names=accepted_slice_names,
    probability_column=PROB_COL,
    threshold=0.5,
    minimum_slice_size=50
)

figure5_slice_results = figure5_slice_results.sort_values(
    "AUPRC",
    ascending=True
).reset_index(drop=True)

figure5_slice_results

In [ ]:
figure5_slice_results = figure5_slice_results.merge(
    accepted_registry[
        ["slice_name", "slice_type_manual"]
    ],
    on="slice_name",
    how="left"
)

figure5_slice_results[
    "slice_type_manual"
] = figure5_slice_results[
    "slice_type_manual"
].fillna("unspecified")

figure5_slice_results.head()

In [ ]:
#from matplotlib.gridspec import GridSpec
#
## ============================================================
## Fig. 5
## Relationship between global predictive performance and
## hidden subgroup evaluation
## ============================================================
#
#fig = plt.figure(figsize=(13.5, 9))
#
#grid = GridSpec(
#    nrows=2,
#    ncols=2,
#    figure=fig,
#    height_ratios=[1.0, 1.55],
#    hspace=0.38,
#    wspace=0.28
#)
#
## ------------------------------------------------------------
## Panel A: Global ROC curve
## ------------------------------------------------------------
#ax_roc = fig.add_subplot(grid[0, 0])
#
#ax_roc.plot(
#    fpr,
#    tpr,
#    linewidth=2,
#    label=f"Baseline model (AUROC = {global_auroc:.3f})"
#)
#
#ax_roc.plot(
#    [0, 1],
#    [0, 1],
#    linestyle="--",
#    linewidth=1,
#    label="Random ranking"
#)
#
#ax_roc.set_xlabel("False Positive Rate")
#ax_roc.set_ylabel("True Positive Rate")
#ax_roc.set_title("(A) Global ROC Evaluation")
#ax_roc.set_xlim(0, 1)
#ax_roc.set_ylim(0, 1.02)
#ax_roc.legend(loc="lower right", frameon=False)
#ax_roc.grid(alpha=0.2)
#
#
## ------------------------------------------------------------
## Panel B: Global precision–recall curve
## ------------------------------------------------------------
#ax_pr = fig.add_subplot(grid[0, 1])
#
#ax_pr.plot(
#    recall,
#    precision,
#    linewidth=2,
#    label=f"Baseline model (AUPRC = {global_auprc:.3f})"
#)
#
#ax_pr.axhline(
#    positive_prevalence,
#    linestyle="--",
#    linewidth=1,
#    label=f"No-skill baseline = {positive_prevalence:.3f}"
#)
#
#ax_pr.set_xlabel("Recall")
#ax_pr.set_ylabel("Precision")
#ax_pr.set_title("(B) Global Precision–Recall Evaluation")
#ax_pr.set_xlim(0, 1)
#ax_pr.set_ylim(0, 1.02)
#ax_pr.legend(loc="lower left", frameon=False)
#ax_pr.grid(alpha=0.2)
#
#
## ------------------------------------------------------------
## Panel C: Slice-level hidden degradation
## ------------------------------------------------------------
#ax_slice = fig.add_subplot(grid[1, :])
#
#plot_df = figure5_slice_results.copy()
#
#y_positions = np.arange(len(plot_df))
#
## Horizontal guide lines from zero to slice AUPRC
#for position, value in zip(
#    y_positions,
#    plot_df["AUPRC"]
#):
#    ax_slice.hlines(
#        y=position,
#        xmin=0,
#        xmax=value,
#        linewidth=1,
#        alpha=0.45
#    )
#
## Slice AUPRC points
#ax_slice.scatter(
#    plot_df["AUPRC"],
#    y_positions,
#    s=55,
#    label="Slice AUPRC",
#    zorder=3
#)
#
## Slice recall points
#ax_slice.scatter(
#    plot_df["Recall"],
#    y_positions,
#    marker="x",
#    s=55,
#    label="Slice Recall@0.5",
#    zorder=4
#)
#
## Global performance references
#ax_slice.axvline(
#    global_auprc,
#    linestyle="--",
#    linewidth=1.5,
#    label=f"Global AUPRC = {global_auprc:.3f}"
#)
#
#global_predicted = (y_prob >= 0.5).astype(int)
#global_recall = recall_score(
#    y_true,
#    global_predicted,
#    zero_division=0
#)
#
#ax_slice.axvline(
#    global_recall,
#    linestyle=":",
#    linewidth=1.5,
#    label=f"Global Recall@0.5 = {global_recall:.3f}"
#)
#
#ax_slice.set_yticks(y_positions)
#ax_slice.set_yticklabels(
#    plot_df["slice_name"]
#)
#
#ax_slice.set_xlabel("Performance")
#ax_slice.set_ylabel("Accepted Latent Slice")
#ax_slice.set_title(
#    "(C) Hidden Slice-Level Degradation Masked by Aggregate Performance"
#)
#
#ax_slice.set_xlim(0, 1)
#ax_slice.grid(
#    axis="x",
#    alpha=0.2
#)
#
#ax_slice.legend(
#    loc="lower right",
#    frameon=False,
#    ncol=2
#)
#
## Highlight the lowest-AUPRC slice
#worst_row = plot_df.iloc[0]
#worst_position = 0
#
#ax_slice.annotate(
#    f"Worst slice:\n{worst_row['slice_name']}\n"
#    f"AUPRC = {worst_row['AUPRC']:.3f}",
#    xy=(
#        worst_row["AUPRC"],
#        worst_position
#    ),
#    xytext=(
#        min(worst_row["AUPRC"] + 0.16, 0.82),
#        worst_position + 1.2
#    ),
#    arrowprops={
#        "arrowstyle": "->",
#        "linewidth": 1
#    },
#    fontsize=9
#)
#
## Overall title
#fig.suptitle(
#    "Global Predictive Performance Can Conceal Hidden Subgroup Failure",
#    fontsize=15,
#    y=0.98
#)
#
#plt.savefig(
#    "Fig5_global_vs_hidden_subgroup_evaluation.png",
#    dpi=600,
#    bbox_inches="tight"
#)
#
#plt.savefig(
#    "Fig5_global_vs_hidden_subgroup_evaluation.pdf",
#    bbox_inches="tight"
#)
#
#plt.show()

In [ ]:
figure5_slice_results.to_csv(
    "Fig5_slice_level_data.csv",
    index=False
)

global_figure5_summary = pd.DataFrame({
    "model_arm": [MODEL_ARM],
    "global_AUROC": [global_auroc],
    "global_AUPRC": [global_auprc],
    "global_Recall_at_0_5": [global_recall],
    "positive_prevalence": [positive_prevalence],
    "number_of_evaluated_slices": [
        len(figure5_slice_results)
    ],
    "worst_slice_name": [
        worst_row["slice_name"]
    ],
    "worst_slice_AUPRC": [
        worst_row["AUPRC"]
    ],
    "worst_slice_Recall_at_0_5": [
        worst_row["Recall"]
    ]
})

global_figure5_summary.to_csv(
    "Fig5_global_summary.csv",
    index=False
)

global_figure5_summary

In [ ]:
import os

for f in os.listdir():
    if f.endswith(".csv"):
        print(f)

## Figure 6
Ranking of Representative Hidden Failure Slices

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

registry = pd.read_csv(
    "approved_slice_registry_FINAL.csv"
)

harm = pd.read_csv(
    "slice_harm.csv",
    index_col=0
)

harm.columns = ["harm_score"]

harm.head()

In [ ]:
plot_df = (
    registry
    .merge(
        harm,
        left_on="slice_name",
        right_index=True,
        how="left"
    )
)

plot_df = plot_df[
    plot_df["final_accept"]
    .astype(str)
    .str.lower()
    .isin(["yes","true","1"])
].copy()

plot_df = plot_df.sort_values(
    "harm_score",
    ascending=False
)

plot_df.reset_index(
    drop=True,
    inplace=True
)

plot_df

In [ ]:
plot_df["colour"] = plot_df[
    "slice_type_manual"
].map({
    "clinical":"tab:blue",
    "hybrid":"tab:orange",
    "operational":"tab:green"
})

plot_df["colour"] = plot_df[
    "colour"
].fillna("gray")

In [ ]:
## ============================================================
## Fig. 6
## Ranking of Representative Hidden Failure Slices
## IEEE publication-quality figure with Pareto curve
## ============================================================
#
#import matplotlib.pyplot as plt
#import numpy as np
#from matplotlib.patches import Patch
#from matplotlib.lines import Line2D
#
## ------------------------------------------------------------
## Colour-blind safe palette (Okabe-Ito)
## ------------------------------------------------------------
#
#COLOR_CLINICAL = "#0072B2"
#COLOR_DUAL = "#D55E00"
#COLOR_OTHER = "#666666"
#TOP3_SHADE = "#FFF4CC"
#
## ------------------------------------------------------------
## Sort by harm
## ------------------------------------------------------------
#
#plot_df = plot_df.copy()
#
#plot_df = plot_df.sort_values(
#    "harm_score",
#    ascending=False
#).reset_index(drop=True)
#
## ------------------------------------------------------------
## Map colours
## ------------------------------------------------------------
#
#plot_df["colour"] = plot_df["slice_type_manual"].map({
#    "clinical": COLOR_CLINICAL,
#    "hybrid": COLOR_DUAL,
#    "operational": COLOR_OTHER
#})
#
#plot_df["colour"] = plot_df["colour"].fillna(COLOR_CLINICAL)
#
## ------------------------------------------------------------
## Cumulative harm (Pareto)
## ------------------------------------------------------------
#
#plot_df["harm_fraction"] = (
#    plot_df["harm_score"] /
#    plot_df["harm_score"].sum()
#)
#
#plot_df["cumulative_harm"] = (
#    plot_df["harm_fraction"]
#    .cumsum()
#)
#
## ------------------------------------------------------------
## Discovery labels
## ------------------------------------------------------------
#
#labels = []
#
#for _, row in plot_df.iterrows():
#
#    if row["slice_type_manual"] == "hybrid":
#        labels.append(f"(D) {row['slice_name']}")
#    else:
#        labels.append(f"(C) {row['slice_name']}")
#
## ------------------------------------------------------------
## Create figure
## ------------------------------------------------------------
#
#fig, ax = plt.subplots(figsize=(12,7))
#
## top x-axis for Pareto curve
#ax2 = ax.twiny()
#
## ------------------------------------------------------------
## Highlight Top 3
## ------------------------------------------------------------
#
#for i in range(min(3, len(plot_df))):
#
#    ax.axhspan(
#        i-0.45,
#        i+0.45,
#        color=TOP3_SHADE,
#        alpha=0.65,
#        zorder=0
#    )
#
## ------------------------------------------------------------
## Horizontal bars
## ------------------------------------------------------------
#
#bars = ax.barh(
#
#    np.arange(len(plot_df)),
#
#    plot_df["harm_score"],
#
#    color=plot_df["colour"],
#
#    edgecolor="black",
#
#    linewidth=0.8,
#
#    zorder=3
#
#)
#
#ax.set_yticks(np.arange(len(plot_df)))
#ax.set_yticklabels(labels)
#
#ax.invert_yaxis()
#
## ------------------------------------------------------------
## Bar values
## ------------------------------------------------------------
#
#for bar in bars:
#
#    width = bar.get_width()
#
#    ax.text(
#
#        width + 0.003,
#
#        bar.get_y() + bar.get_height()/2,
#
#        f"{width:.3f}",
#
#        va="center",
#
#        fontsize=9
#
#    )
#
## ------------------------------------------------------------
## Top 3 annotations
## ------------------------------------------------------------
#
#annotation_text = [
#
#    "Top Repair Candidate",
#
#    "Second Highest Harm",
#
#    "Third Highest Harm"
#
#]
#
#for i in range(min(3, len(plot_df))):
#
#    row = plot_df.iloc[i]
#
#    ax.scatter(
#
#        row["harm_score"],
#
#        i,
#
#        marker="*",
#
#        s=180,
#
#        color="gold",
#
#        edgecolor="black",
#
#        zorder=6
#
#    )
#
#    ax.annotate(
#
#        annotation_text[i],
#
#        xy=(row["harm_score"], i),
#
#        xytext=(row["harm_score"] + 0.04, i - 0.22),
#
#        arrowprops=dict(
#            arrowstyle="->",
#            lw=0.8
#        ),
#
#        fontsize=9
#
#    )
#
## ------------------------------------------------------------
## Pareto cumulative harm curve
## ------------------------------------------------------------
#
#ax2.plot(
#
#    plot_df["cumulative_harm"],
#
#    np.arange(len(plot_df)),
#
#    color="black",
#
#    linewidth=2,
#
#    marker="o",
#
#    markersize=5,
#
#    zorder=8
#
#)
#
#ax2.axvline(
#
#    0.80,
#
#    linestyle="--",
#
#    color="gray",
#
#    linewidth=1.2
#
#)
#
#ax2.text(
#
#    0.805,
#
#    len(plot_df)-0.8,
#
#    "80%",
#
#    fontsize=8,
#
#    color="gray"
#
#)
#
#ax2.set_xlim(0,1)
#
#ax2.set_xticks(np.linspace(0,1,6))
#
#ax2.set_xticklabels(
#
#    ["0%","20%","40%","60%","80%","100%"]
#
#)
#
#ax2.set_xlabel(
#
#    "Cumulative Harm Proportion",
#
#    fontsize=11
#
#)
#
## ------------------------------------------------------------
## Axis labels
## ------------------------------------------------------------
#
#ax.set_xlabel(
#
#    "BASiC Composite Harm Score",
#
#    fontsize=11
#
#)
#
#ax.set_ylabel(
#
#    "Accepted Hidden Failure Slice",
#
#    fontsize=11
#
#)
#
#ax.set_title(
#
#    "Representative Hidden Failure Slices Ranked by BASiC Composite Harm Score",
#
#    fontsize=13,
#
#    weight="bold"
#
#)
#
#ax.grid(
#
#    axis="x",
#
#    alpha=0.25
#
#)
#
## ------------------------------------------------------------
## Figure note
## ------------------------------------------------------------
##
##x.text(
##
##   0.02,
##
##   len(plot_df)-0.3,
##
##   "The Pareto curve illustrates that a small number\n"
##   "of clinically interpretable slices account for\n"
##   "most of the observed hidden prediction harm.",
##
##   fontsize=8.5,
##
##   bbox=dict(
##
##       facecolor="white",
##
##       edgecolor="gray",
##
##       alpha=0.9
##
##   )
##
##
#
## ------------------------------------------------------------
## Legend
## ------------------------------------------------------------
#
#legend_elements = [
#
#    Patch(
#
#        facecolor=COLOR_DUAL,
#
#        edgecolor="black",
#
#        label="Dual-supported"
#
#    ),
#
#    Patch(
#
#        facecolor=COLOR_CLINICAL,
#
#        edgecolor="black",
#
#        label="Clinically supported"
#
#    ),
#
#    Patch(
#
#        facecolor=TOP3_SHADE,
#
#        edgecolor="black",
#
#        label="Top 3 repair candidates"
#
#    ),
#
#    Line2D(
#
#        [0],
#
#        [0],
#
#        color="black",
#
#        marker="o",
#
#        linewidth=2,
#
#        label="Cumulative harm"
#
#    )
#
#]
#
#ax.legend(
#
#    handles=legend_elements,
#
#    frameon=False,
#
#    loc="lower right"
#
#)
#
## ------------------------------------------------------------
## Save BEFORE show
## ------------------------------------------------------------
#
#plt.tight_layout()
#
#fig.savefig(
#    "Fig6_BASiC_Harm_Ranking.pdf",
#    dpi=600,
#    bbox_inches="tight"
#)
#
#fig.savefig(
#    "Fig6_BASiC_Harm_Ranking.png",
#    dpi=600,
#    bbox_inches="tight"
#)
#
#plt.show()
#
#plt.close(fig)

In [ ]:
# ============================================================
# FIGURE 4 — DECOMPOSITION OF COMPOSITE OPERATIONAL HARM
# FINAL IEEE PUBLICATION VERSION
#
# Scientific purpose:
#   Figure 3 -> demonstrates local predictive heterogeneity.
#   Figure 4 -> explains WHY the positive-harm slices are harmful.
#
# Operational harm:
#
#       H_k = ΔL_k + ΔFN_k + ΔHCC_k
#
# where:
#   ΔL_k   = slice log-loss - global log-loss
#   ΔFN_k  = slice FN rate - global FN rate
#   ΔHCC_k = slice HCC rate - global HCC rate
#
# Figure design:
#   - ALL positive-harm BASiC slices (H_k > 0)
#   - diverging stacked horizontal bars
#   - component colours encode ΔL / ΔFN / ΔHCC
#   - marker SHAPE encodes discovery provenance:
#         ◆ dual-supported
#         ○ clinically supported
#   - total H_k displayed separately
#   - zero reference line
#   - no Pareto curve
#   - no BASiCRank / priority score
#   - no governance score
#   - PDF + SVG vector exports
#   - PNG 600 dpi export
#
# FINAL LAYOUT REFINEMENTS:
#   - increased separation between title and equation
#   - increased separation between the two legend rows and note
# ============================================================

import os
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")
plt.close("all")

# ============================================================
# 0. OUTPUT DIRECTORY
# ============================================================

FIG_DIR = Path.cwd() / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

PDF_PATH = FIG_DIR / "Fig4_Operational_Harm_Decomposition.pdf"
SVG_PATH = FIG_DIR / "Fig4_Operational_Harm_Decomposition.svg"
PNG_PATH = FIG_DIR / "Fig4_Operational_Harm_Decomposition.png"
DATA_PATH = FIG_DIR / "Fig4_Operational_Harm_Decomposition_Data.csv"

# ============================================================
# 1. IEEE-FRIENDLY TYPOGRAPHY
# ============================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "axes.linewidth": 0.8,
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def canonical_slice_name(value):

    text = str(value).strip().lower()

    text = re.sub(
        r"[\s\-/]+",
        "_",
        text
    )

    text = re.sub(
        r"_+",
        "_",
        text
    )

    return text.strip("_")


def pretty_slice_name(value):

    text = canonical_slice_name(value)

    text = text.replace(
        "_and_",
        " + "
    )

    text = text.replace(
        "_",
        " "
    )

    replacements = {
        "hr missing": "HR missing",
        "spo2 missing": "SpO₂ missing",
        "ckd": "CKD",
        "chd": "CHD",
        "dm": "DM",
        "htn": "HTN",
        "vitals missing ge 3": "vitals missing ≥ 3"
    }

    for old, new in replacements.items():

        text = re.sub(
            rf"\b{re.escape(old)}\b",
            new,
            text,
            flags=re.IGNORECASE
        )

    if text:
        text = text[0].upper() + text[1:]

    return text


def find_column(
    df,
    exact_candidates,
    keyword_groups=None
):

    # Exact match
    for candidate in exact_candidates:

        if candidate in df.columns:
            return candidate

    # Case-insensitive exact match
    lookup = {
        str(c).strip().lower(): c
        for c in df.columns
    }

    for candidate in exact_candidates:

        key = (
            str(candidate)
            .strip()
            .lower()
        )

        if key in lookup:
            return lookup[key]

    # Conservative keyword matching
    if keyword_groups:

        for keywords in keyword_groups:

            matches = []

            for c in df.columns:

                cl = str(c).lower()

                if all(
                    keyword.lower() in cl
                    for keyword in keywords
                ):
                    matches.append(c)

            if len(matches) == 1:
                return matches[0]

    return None


def as_bool(series):

    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)

    text = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    return text.isin({
        "true",
        "1",
        "yes",
        "y",
        "dual",
        "dual_supported",
        "dual_supported_overlap"
    })


# ============================================================
# 3. LOCATE COMPONENT SOURCE
#
# Figure 4 requires the actual three diagnostic components.
# ============================================================

component_candidates = [
    "nb8_composite_harm_components.csv",
    "nb8_asap_candidate_pool_with_ablations.csv",
    "nb8_asap_complete_candidate_audit.csv",
    "nb8_slice_source_audit.csv"
]

component_file = None
component_df = None

for candidate in component_candidates:

    if not Path(candidate).exists():
        continue

    temp = pd.read_csv(candidate)

    temp_slice = find_column(
        temp,
        [
            "slice_name",
            "slice",
            "slice_id",
            "name"
        ]
    )

    temp_loss = find_column(
        temp,
        [
            "loss_lift",
            "logloss_lift",
            "val_logloss_lift",
            "delta_loss",
            "delta_l",
            "dL",
            "ΔL"
        ],
        [
            ["loss", "lift"]
        ]
    )

    temp_fn = find_column(
        temp,
        [
            "fn_lift",
            "false_negative_lift",
            "val_fn_lift",
            "delta_fn",
            "dFN",
            "ΔFN"
        ],
        [
            ["fn", "lift"],
            ["false", "negative", "lift"]
        ]
    )

    temp_hcc = find_column(
        temp,
        [
            "hcc_lift",
            "high_confidence_contradiction_lift",
            "val_hcc_lift",
            "delta_hcc",
            "dHCC",
            "ΔHCC"
        ],
        [
            ["hcc", "lift"],
            ["confidence", "contradiction", "lift"]
        ]
    )

    if (
        temp_slice is not None
        and temp_loss is not None
        and temp_fn is not None
        and temp_hcc is not None
    ):

        component_file = candidate
        component_df = temp

        slice_col = temp_slice
        loss_col = temp_loss
        fn_col = temp_fn
        hcc_col = temp_hcc

        break


if component_file is None:

    raise ValueError(
        "\nFigure 4 requires ΔL, ΔFN and ΔHCC for each slice, "
        "but no available CSV contained all three components.\n\n"
        "Checked:\n  - "
        + "\n  - ".join(component_candidates)
        + "\n\nDo not substitute priority_score or normalized harm."
    )


print("=" * 78)
print("FIGURE 4 — OPERATIONAL-HARM DECOMPOSITION")
print("=" * 78)

print("\nComponent source:")
print(" ", component_file)

print("\nDetected columns:")
print("  Slice :", slice_col)
print("  ΔL    :", loss_col)
print("  ΔFN   :", fn_col)
print("  ΔHCC  :", hcc_col)

# ============================================================
# 4. BUILD COMPONENT TABLE
# ============================================================

harm = component_df[
    [
        slice_col,
        loss_col,
        fn_col,
        hcc_col
    ]
].copy()

harm.columns = [
    "slice_name",
    "delta_L",
    "delta_FN",
    "delta_HCC"
]

harm["slice_key"] = (
    harm["slice_name"]
    .apply(canonical_slice_name)
)

for column in [
    "delta_L",
    "delta_FN",
    "delta_HCC"
]:

    harm[column] = pd.to_numeric(
        harm[column],
        errors="coerce"
    )

harm = (
    harm
    .dropna(
        subset=[
            "slice_key",
            "delta_L",
            "delta_FN",
            "delta_HCC"
        ]
    )
    .drop_duplicates(
        subset="slice_key",
        keep="first"
    )
)

# ============================================================
# 5. COMPUTE TOTAL OPERATIONAL HARM
# ============================================================

harm["H_k"] = (
    harm["delta_L"]
    + harm["delta_FN"]
    + harm["delta_HCC"]
)

# Positive-harm slices only
harm = harm[
    harm["H_k"] > 0
].copy()

if harm.empty:

    raise ValueError(
        "No positive-harm slices were found after calculating "
        "H_k = ΔL + ΔFN + ΔHCC."
    )

print(
    "\nPositive-harm slices identified:",
    len(harm)
)

# ============================================================
# 6. LOAD VERIFIED DISCOVERY PROVENANCE
#
# CRITICAL:
# nb8_dual_source_combined_slices.csv is NOT used as a
# dual-supported registry.
# ============================================================

provenance_file = "nb8_slice_source_audit.csv"

provenance = None

if Path(provenance_file).exists():

    provenance_raw = pd.read_csv(
        provenance_file
    )

    prov_slice_col = find_column(
        provenance_raw,
        [
            "slice_name",
            "slice",
            "slice_id",
            "name"
        ]
    )

    dual_col = find_column(
        provenance_raw,
        [
            "has_dual_evidence",
            "dual_supported",
            "dual_evidence"
        ]
    )

    evidence_col = find_column(
        provenance_raw,
        [
            "discovery_evidence",
            "evidence",
            "source_evidence"
        ]
    )

    if prov_slice_col is not None:

        provenance = pd.DataFrame({
            "slice_name_prov":
                provenance_raw[prov_slice_col]
        })

        provenance["slice_key"] = (
            provenance["slice_name_prov"]
            .apply(canonical_slice_name)
        )

        if dual_col is not None:

            provenance["is_dual"] = as_bool(
                provenance_raw[dual_col]
            )

        elif evidence_col is not None:

            provenance["is_dual"] = (
                provenance_raw[evidence_col]
                .astype(str)
                .str.lower()
                .str.contains(
                    "dual",
                    na=False
                )
            )

        else:
            provenance = None


# ============================================================
# 7. FALLBACK TO TRUE DUAL-SUPPORTED FILE
# ============================================================

if provenance is None:

    dual_file = "nb8_dual_supported_slices.csv"

    if Path(dual_file).exists():

        dual_raw = pd.read_csv(
            dual_file
        )

        dual_slice_col = find_column(
            dual_raw,
            [
                "slice_name",
                "slice",
                "slice_id",
                "name"
            ]
        )

        if dual_slice_col is None:

            raise ValueError(
                "nb8_dual_supported_slices.csv exists but "
                "contains no identifiable slice-name column."
            )

        dual_set = set(
            dual_raw[dual_slice_col]
            .dropna()
            .astype(str)
            .apply(canonical_slice_name)
        )

        provenance = pd.DataFrame({
            "slice_key": harm["slice_key"]
        })

        provenance["is_dual"] = (
            provenance["slice_key"]
            .isin(dual_set)
        )

    else:

        # Verified Notebook 8 fallback
        verified_dual = {
            "abdominal_pain_and_hr_missing",
            "ckd_and_spo2_missing",
            "unclassified_and_vitals_missing_ge_3"
        }

        provenance = pd.DataFrame({
            "slice_key": harm["slice_key"]
        })

        provenance["is_dual"] = (
            provenance["slice_key"]
            .isin(verified_dual)
        )


provenance = (
    provenance[
        [
            "slice_key",
            "is_dual"
        ]
    ]
    .drop_duplicates(
        subset="slice_key",
        keep="first"
    )
)

# ============================================================
# 8. MERGE HARM AND PROVENANCE
# ============================================================

plot_df = harm.merge(
    provenance,
    on="slice_key",
    how="left"
)

plot_df["is_dual"] = (
    plot_df["is_dual"]
    .fillna(False)
    .astype(bool)
)

plot_df["provenance"] = np.where(
    plot_df["is_dual"],
    "Dual-supported",
    "Clinically supported"
)

plot_df["display_name"] = (
    plot_df["slice_key"]
    .apply(pretty_slice_name)
)

# ============================================================
# 9. VALIDATE PROVENANCE
# ============================================================

print("\nDiscovery provenance counts:")

print(
    plot_df["provenance"]
    .value_counts()
)

dual_detected = (
    plot_df.loc[
        plot_df["is_dual"],
        "slice_key"
    ]
    .tolist()
)

print("\nDual-supported slices shown:")

for s in dual_detected:
    print("  ◆", s)

if len(dual_detected) > 3:

    raise ValueError(
        "\nMore than three dual-supported positive-harm slices "
        "were detected. Inspect nb8_slice_source_audit.csv "
        "before using Figure 4."
    )

# ============================================================
# 10. SORT BY TOTAL H_k
#
# Highest harm appears at the TOP.
# ============================================================

plot_df = (
    plot_df
    .sort_values(
        "H_k",
        ascending=False
    )
    .reset_index(drop=True)
)

# ============================================================
# 11. EXPORT EXACT SOURCE DATA
# ============================================================

plot_df[
    [
        "slice_name",
        "provenance",
        "delta_L",
        "delta_FN",
        "delta_HCC",
        "H_k"
    ]
].to_csv(
    DATA_PATH,
    index=False
)

# ============================================================
# 12. COMPONENT COLOURS
# ============================================================

COLOR_LOSS = "#0072B2"
COLOR_FN = "#D55E00"
COLOR_HCC = "#009E73"

# ============================================================
# 13. FIGURE SIZE
#
# Slightly taller than previous version to create more
# breathing room around legends and footnote.
# ============================================================

n_slices = len(plot_df)

figure_height = max(
    5.25,
    0.39 * n_slices + 2.0
)

fig, ax = plt.subplots(
    figsize=(
        7.16,
        figure_height
    )
)

y = np.arange(
    n_slices
)

# ============================================================
# 14. DRAW DIVERGING STACKED BARS
# ============================================================

components = [
    (
        "delta_L",
        r"$\Delta L_k$",
        COLOR_LOSS
    ),
    (
        "delta_FN",
        r"$\Delta FN_k$",
        COLOR_FN
    ),
    (
        "delta_HCC",
        r"$\Delta HCC_k$",
        COLOR_HCC
    )
]

positive_base = np.zeros(
    n_slices,
    dtype=float
)

negative_base = np.zeros(
    n_slices,
    dtype=float
)

for column, label, color in components:

    values = (
        plot_df[column]
        .astype(float)
        .to_numpy()
    )

    positive_values = np.where(
        values > 0,
        values,
        0
    )

    negative_values = np.where(
        values < 0,
        values,
        0
    )

    ax.barh(
        y,
        positive_values,
        left=positive_base,
        height=0.60,
        color=color,
        edgecolor="white",
        linewidth=0.55,
        label=label
    )

    ax.barh(
        y,
        negative_values,
        left=negative_base,
        height=0.60,
        color=color,
        edgecolor="white",
        linewidth=0.55
    )

    positive_base += positive_values
    negative_base += negative_values

# ============================================================
# 15. ZERO REFERENCE
# ============================================================

ax.axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=1.1,
    zorder=5
)

# ============================================================
# 16. TOTAL H_k MARKERS
#
# Diamond = dual-supported
# Circle  = clinically supported
# ============================================================

for i, row in plot_df.iterrows():

    marker = (
        "D"
        if row["is_dual"]
        else "o"
    )

    ax.scatter(
        row["H_k"],
        i,
        marker=marker,
        s=48 if row["is_dual"] else 38,
        facecolor="white",
        edgecolor="black",
        linewidth=1.0,
        zorder=8
    )

# ============================================================
# 17. TOTAL H_k LABELS
# ============================================================

all_extremes = np.concatenate(
    [
        positive_base,
        negative_base,
        plot_df["H_k"].to_numpy()
    ]
)

plot_span = max(
    0.05,
    float(
        all_extremes.max()
        - all_extremes.min()
    )
)

value_offset = max(
    0.006,
    0.025 * plot_span
)

for i, row in plot_df.iterrows():

    h = float(
        row["H_k"]
    )

    ax.text(
        max(
            positive_base[i],
            h
        )
        + value_offset,
        i,
        f"$H_k$={h:.3f}",
        ha="left",
        va="center",
        fontsize=6.7,
        fontweight="bold"
    )

# ============================================================
# 18. Y-AXIS LABELS + PROVENANCE SYMBOLS
# ============================================================

ytick_labels = []

for _, row in plot_df.iterrows():

    prefix = (
        "◆ "
        if row["is_dual"]
        else "○ "
    )

    ytick_labels.append(
        prefix
        + row["display_name"]
    )

ax.set_yticks(
    y
)

ax.set_yticklabels(
    ytick_labels,
    fontsize=7.1
)

ax.invert_yaxis()

# ============================================================
# 19. X-LIMITS
# ============================================================

xmin = min(
    float(
        negative_base.min()
    ),
    0
)

xmax = max(
    float(
        positive_base.max()
    ),
    float(
        plot_df["H_k"].max()
    )
)

xspan = max(
    0.05,
    xmax - xmin
)

ax.set_xlim(
    xmin - 0.06 * xspan,
    xmax + 0.24 * xspan
)

# ============================================================
# 20. TITLE AND EQUATION
#
# FINAL FIX #1:
# Greater separation between title and equation.
# ============================================================

ax.set_xlabel(
    "Slice-vs-global diagnostic lift"
)

ax.set_title(
    "Decomposition of composite operational harm across positive-harm BASiC candidates",
    loc="left",
    fontsize=9,
    fontweight="bold",
    pad=27
)

# Equation now has its own visual row below the title
ax.text(
    0.0,
    1.012,
    r"$H_k = \Delta L_k + \Delta FN_k + \Delta HCC_k$",
    transform=ax.transAxes,
    ha="left",
    va="bottom",
    fontsize=7.0,
    color="0.20"
)

# ============================================================
# 21. GRID / SPINES
# ============================================================

ax.grid(
    axis="x",
    linestyle=":",
    linewidth=0.55,
    alpha=0.35
)

ax.set_axisbelow(True)

ax.spines[
    ["top", "right"]
].set_visible(False)

# ============================================================
# 22. COMPONENT LEGEND
# ============================================================

component_handles = [

    Patch(
        facecolor=COLOR_LOSS,
        edgecolor="none",
        label=r"Excess log-loss ($\Delta L_k$)"
    ),

    Patch(
        facecolor=COLOR_FN,
        edgecolor="none",
        label=r"FN lift ($\Delta FN_k$)"
    ),

    Patch(
        facecolor=COLOR_HCC,
        edgecolor="none",
        label=r"HCC lift ($\Delta HCC_k$)"
    )

]

provenance_handles = [

    Line2D(
        [0],
        [0],
        marker="D",
        linestyle="",
        markerfacecolor="white",
        markeredgecolor="black",
        markersize=5.5,
        label="Dual-supported"
    ),

    Line2D(
        [0],
        [0],
        marker="o",
        linestyle="",
        markerfacecolor="white",
        markeredgecolor="black",
        markersize=5.5,
        label="Clinically supported"
    )

]

# ------------------------------------------------------------
# FINAL FIX #2:
# Separate the two legend rows more clearly.
# ------------------------------------------------------------

legend1 = ax.legend(
    handles=component_handles,
    loc="upper center",
    bbox_to_anchor=(
        0.50,
        -0.095
    ),
    frameon=False,
    ncol=3,
    columnspacing=1.5,
    handletextpad=0.55,
    fontsize=6.7
)

ax.add_artist(
    legend1
)

ax.legend(
    handles=provenance_handles,
    loc="upper center",
    bbox_to_anchor=(
        0.50,
        -0.180
    ),
    frameon=False,
    ncol=2,
    columnspacing=2.2,
    handletextpad=0.65,
    fontsize=6.7
)

# ============================================================
# 23. SHORT FIGURE NOTE
#
# FINAL FIX #2 continued:
# Note pushed farther below the provenance legend.
# ============================================================

fig.text(
    0.5,
    0.016,
    (
        "Bar segments quantify diagnostic contributions to operational harm; "
        "provenance markers indicate discovery evidence only. "
        "ASAP governance is assessed separately."
    ),
    ha="center",
    va="bottom",
    fontsize=6.8
)

# ============================================================
# 24. FINAL LAYOUT
#
# Extra top + bottom breathing room.
# ============================================================

fig.subplots_adjust(
    left=0.31,
    right=0.97,
    top=0.875,
    bottom=0.245
)

# ============================================================
# 25. EXPORT
# ============================================================

fig.savefig(
    PDF_PATH,
    format="pdf",
    bbox_inches="tight",
    pad_inches=0.10
)

fig.savefig(
    SVG_PATH,
    format="svg",
    bbox_inches="tight",
    pad_inches=0.10
)

fig.savefig(
    PNG_PATH,
    format="png",
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.10
)

# ============================================================
# 26. VERIFY FILE CREATION
# ============================================================

print("\n" + "=" * 78)
print("FIGURE 4 EXPORT VERIFICATION")
print("=" * 78)

for label, path in {
    "PDF": PDF_PATH,
    "SVG": SVG_PATH,
    "PNG": PNG_PATH,
    "DATA": DATA_PATH
}.items():

    if not path.exists():

        raise IOError(
            f"{label} was not created: {path}"
        )

    size_kb = (
        path.stat().st_size
        / 1024
    )

    if size_kb <= 0:

        raise IOError(
            f"{label} appears empty: {path}"
        )

    print(
        f"{label}: {path.resolve()}"
    )

    print(
        f"     size = {size_kb:.1f} KB"
    )

plt.show()
plt.close(fig)

# ============================================================
# 27. REPORT EXACT DATA USED
# ============================================================

print("\n" + "=" * 78)
print("FIGURE 4 GENERATED SUCCESSFULLY")
print("=" * 78)

print("\nOperational-harm definition:")
print("H_k = ΔL_k + ΔFN_k + ΔHCC_k")

print(
    "\nPositive-harm slices displayed:",
    len(plot_df)
)

print(
    "\nDual-supported slices displayed:",
    int(plot_df["is_dual"].sum())
)

print(
    "\nClinically supported slices displayed:",
    int((~plot_df["is_dual"]).sum())
)

print("\nExact Figure 4 data:")

display(
    plot_df[
        [
            "slice_name",
            "provenance",
            "delta_L",
            "delta_FN",
            "delta_HCC",
            "H_k"
        ]
    ]
    .style
    .format({
        "delta_L": "{:.4f}",
        "delta_FN": "{:.4f}",
        "delta_HCC": "{:.4f}",
        "H_k": "{:.4f}"
    })
)

print("\nInterpretation:")
print("Bar colour = diagnostic mechanism.")
print("Marker shape = discovery provenance.")
print(
    "Bar magnitude does NOT encode ASAP acceptance, "
    "governance score, or repair priority."
)

print("\nFigure directory:")
print(FIG_DIR.resolve())

print("=" * 78)

## Figure 7
Comparative Evaluation of Slice Discovery Methods

In [ ]:
# ============================================================
# FIGURE 5 — COMPARATIVE DISCOVERY PROFILE
# FINAL IEEE PUBLICATION VERSION
#
# Panel (a): Statistical discovery behaviour
#   - Mean composite harm
#   - Maximum composite harm
#   - Worst-slice AUPRC severity
#   - True union coverage
#
# Panel (b): Complete ASAP qualification
#   - BASiC / ASAP     = 5/10  = 50%
#   - SliceFinder      = 3/10  = 30%
#   - SliceLine        = 3/10  = 30%
#   - AFISP-inspired   = 0/5   = 0%
#
# FINAL POLISH:
#   - y-axis renamed to "Normalized discovery characteristic"
#   - normalization note moved away from Panel (a) title
#   - more spacing added below Panel (a) legend
#   - PDF / SVG vector export + PNG 600 dpi
# ============================================================

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.close("all")

# ============================================================
# 0. OUTPUT DIRECTORY
# ============================================================

FIG_DIR = Path.cwd() / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

PDF_PATH = FIG_DIR / "Fig5_Comparative_Discovery_Profile.pdf"
SVG_PATH = FIG_DIR / "Fig5_Comparative_Discovery_Profile.svg"
PNG_PATH = FIG_DIR / "Fig5_Comparative_Discovery_Profile.png"

RAW_DATA_PATH = (
    FIG_DIR / "Fig5_Comparative_Discovery_Profile_Raw.csv"
)

NORMALIZED_DATA_PATH = (
    FIG_DIR / "Fig5_Comparative_Discovery_Profile_Normalized.csv"
)

ASAP_DATA_PATH = (
    FIG_DIR / "Fig5_Complete_ASAP_Qualification.csv"
)

# ============================================================
# 1. IEEE-FRIENDLY TYPOGRAPHY
# ============================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "axes.linewidth": 0.8,
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# ============================================================
# 2. HELPERS
# ============================================================

def standardize_columns(df):

    df = df.copy()

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[\s\-/]+", "_", regex=True)
        .str.replace(r"[^a-z0-9_]", "", regex=True)
        .str.replace(r"_+", "_", regex=True)
        .str.strip("_")
    )

    return df


def find_column(
    df,
    exact_candidates,
    keyword_groups=None,
    required=True,
    description=""
):

    for candidate in exact_candidates:
        if candidate in df.columns:
            return candidate

    if keyword_groups:

        for keywords in keyword_groups:

            matches = []

            for c in df.columns:

                cl = str(c).lower()

                if all(
                    keyword.lower() in cl
                    for keyword in keywords
                ):
                    matches.append(c)

            if len(matches) == 1:
                return matches[0]

    if required:

        raise ValueError(
            f"\nCould not identify '{description}'.\n"
            f"Accepted names: {exact_candidates}\n"
            f"Available columns:\n{list(df.columns)}"
        )

    return None


def canonical_method_name(value):

    text = (
        str(value)
        .strip()
        .lower()
        .replace("-", "_")
        .replace("/", "_")
        .replace(" ", "_")
    )

    if "basic" in text or "asap" in text:
        return "BASiC / ASAP"

    if "afisp" in text:
        return "AFISP-inspired"

    if "slicefinder" in text or "slice_finder" in text:
        return "SliceFinder-style"

    if "sliceline" in text or "slice_line" in text:
        return "SliceLine-style"

    return str(value)


def minmax(series):

    x = pd.to_numeric(
        series,
        errors="coerce"
    ).astype(float)

    xmin = x.min()
    xmax = x.max()

    if np.isclose(xmax, xmin):

        return pd.Series(
            0.5,
            index=x.index
        )

    return (
        (x - xmin)
        /
        (xmax - xmin)
    )

# ============================================================
# 3. LOCATE COMPARATIVE SUMMARY
# ============================================================

summary_candidates = [
    "table_X_comparative_method_summary.csv",
    "comparative_summary.csv",
    "05b_comparative_summary_complete_ASAP.csv"
]

summary_file = next(
    (
        f
        for f in summary_candidates
        if Path(f).exists()
    ),
    None
)

if summary_file is None:

    raise FileNotFoundError(
        "\nCould not locate comparative summary data.\n\n"
        "Expected one of:\n  - "
        + "\n  - ".join(summary_candidates)
    )

comparison = pd.read_csv(
    summary_file
)

comparison = standardize_columns(
    comparison
)

print("=" * 78)
print("FIGURE 5 — COMPARATIVE DISCOVERY PROFILE")
print("=" * 78)

print("\nComparative source:")
print(" ", summary_file)

# ============================================================
# 4. IDENTIFY REQUIRED COLUMNS
# ============================================================

method_col = find_column(
    comparison,
    [
        "method",
        "method_name",
        "discovery_method",
        "slice_discovery_method"
    ],
    description="method"
)

mean_harm_col = find_column(
    comparison,
    [
        "mean_harm",
        "mean_composite_harm",
        "mean_harm_score",
        "mean_nb8_harm",
        "average_harm",
        "avg_harm"
    ],
    [
        ["mean", "harm"]
    ],
    description="mean composite harm"
)

max_harm_col = find_column(
    comparison,
    [
        "max_harm",
        "maximum_harm",
        "max_composite_harm",
        "maximum_composite_harm",
        "max_harm_score",
        "max_nb8_harm"
    ],
    [
        ["max", "harm"]
    ],
    description="maximum composite harm"
)

worst_auprc_col = find_column(
    comparison,
    [
        "worst_slice_auprc",
        "worst_auprc",
        "minimum_slice_auprc",
        "min_slice_auprc"
    ],
    [
        ["worst", "auprc"],
        ["min", "auprc"]
    ],
    description="worst-slice AUPRC"
)

coverage_col = find_column(
    comparison,
    [
        "true_union_coverage",
        "true_union_coverage_pct",
        "union_coverage",
        "coverage",
        "coverage_pct",
        "coverage_percent",
        "coverage_percentage",
        "pct_patients_covered",
        "patient_coverage"
    ],
    [
        ["union", "coverage"],
        ["coverage"]
    ],
    description="true union coverage"
)

print("\nColumns selected:")
print("  Method             :", method_col)
print("  Mean harm          :", mean_harm_col)
print("  Maximum harm       :", max_harm_col)
print("  Worst-slice AUPRC  :", worst_auprc_col)
print("  True union coverage:", coverage_col)

# ============================================================
# 5. BUILD CLEAN COMPARATIVE TABLE
# ============================================================

profile = comparison[
    [
        method_col,
        mean_harm_col,
        max_harm_col,
        worst_auprc_col,
        coverage_col
    ]
].copy()

profile.columns = [
    "method",
    "mean_harm",
    "max_harm",
    "worst_slice_auprc",
    "true_union_coverage"
]

profile["method"] = (
    profile["method"]
    .apply(canonical_method_name)
)

required_methods = [
    "BASiC / ASAP",
    "SliceFinder-style",
    "SliceLine-style",
    "AFISP-inspired"
]

profile = profile[
    profile["method"].isin(
        required_methods
    )
].copy()

for c in [
    "mean_harm",
    "max_harm",
    "worst_slice_auprc",
    "true_union_coverage"
]:

    profile[c] = pd.to_numeric(
        profile[c],
        errors="coerce"
    )

profile = (
    profile
    .groupby(
        "method",
        as_index=False
    )
    .agg(
        mean_harm=(
            "mean_harm",
            "mean"
        ),
        max_harm=(
            "max_harm",
            "max"
        ),
        worst_slice_auprc=(
            "worst_slice_auprc",
            "min"
        ),
        true_union_coverage=(
            "true_union_coverage",
            "max"
        )
    )
)

missing_methods = [
    method
    for method in required_methods
    if method not in profile["method"].values
]

if missing_methods:

    raise ValueError(
        "\nThe comparative summary does not contain all four "
        "required methods.\n"
        f"Missing: {missing_methods}\n"
        f"Detected: {profile['method'].tolist()}"
    )

# ============================================================
# 6. STANDARDIZE COVERAGE SCALE
# ============================================================

if (
    profile["true_union_coverage"].max() > 1.0
    and
    profile["true_union_coverage"].max() <= 100
):

    profile["true_union_coverage"] = (
        profile["true_union_coverage"]
        / 100.0
    )

profile = (
    profile
    .set_index("method")
    .loc[required_methods]
    .reset_index()
)

profile.to_csv(
    RAW_DATA_PATH,
    index=False
)

# ============================================================
# 7. NORMALIZE PANEL-A METRICS
# ============================================================

profile_norm = profile.copy()

profile_norm["mean_harm_norm"] = minmax(
    profile["mean_harm"]
)

profile_norm["max_harm_norm"] = minmax(
    profile["max_harm"]
)

raw_auprc_norm = minmax(
    profile["worst_slice_auprc"]
)

profile_norm[
    "worst_auprc_severity_norm"
] = (
    1.0
    - raw_auprc_norm
)

profile_norm[
    "coverage_norm"
] = minmax(
    profile["true_union_coverage"]
)

profile_norm.to_csv(
    NORMALIZED_DATA_PATH,
    index=False
)

# ============================================================
# 8. COMPLETE ASAP QUALIFICATION
# ============================================================

asap = pd.DataFrame({
    "method": [
        "BASiC / ASAP",
        "SliceFinder-style",
        "SliceLine-style",
        "AFISP-inspired"
    ],

    "qualified": [
        5,
        3,
        3,
        0
    ],

    "reported": [
        10,
        10,
        10,
        5
    ]
})

asap["qualification_rate"] = (
    asap["qualified"]
    /
    asap["reported"]
)

asap.to_csv(
    ASAP_DATA_PATH,
    index=False
)

# ============================================================
# 9. METHOD COLOURS / MARKERS
# ============================================================

METHOD_COLORS = {
    "BASiC / ASAP": "#D55E00",
    "SliceFinder-style": "#009E73",
    "SliceLine-style": "#CC79A7",
    "AFISP-inspired": "#0072B2"
}

METHOD_MARKERS = {
    "BASiC / ASAP": "o",
    "SliceFinder-style": "^",
    "SliceLine-style": "D",
    "AFISP-inspired": "s"
}

# ============================================================
# 10. CREATE TWO-PANEL FIGURE
# ============================================================

fig = plt.figure(
    figsize=(
        7.16,
        4.90
    )
)

gs = fig.add_gridspec(
    1,
    2,
    width_ratios=[
        1.45,
        1.0
    ],
    wspace=0.39
)

ax1 = fig.add_subplot(
    gs[0, 0]
)

ax2 = fig.add_subplot(
    gs[0, 1]
)

# ============================================================
# PANEL (a) — STATISTICAL DISCOVERY PROFILE
# ============================================================

metric_columns = [
    "mean_harm_norm",
    "max_harm_norm",
    "worst_auprc_severity_norm",
    "coverage_norm"
]

metric_labels = [
    "Mean\nharm",
    "Maximum\nharm",
    "Worst-slice\nAUPRC severity",
    "True union\ncoverage"
]

x = np.arange(
    len(metric_columns)
)

for _, row in profile_norm.iterrows():

    method = row["method"]

    values = (
        row[metric_columns]
        .astype(float)
        .to_numpy()
    )

    ax1.plot(
        x,
        values,
        color=METHOD_COLORS[method],
        marker=METHOD_MARKERS[method],
        markersize=5.8
        if method == "BASiC / ASAP"
        else 5.0,
        linewidth=2.2
        if method == "BASiC / ASAP"
        else 1.55,
        markeredgecolor="black",
        markeredgewidth=0.45,
        label=method,
        zorder=5
        if method == "BASiC / ASAP"
        else 3
    )

# ------------------------------------------------------------
# Panel A formatting
# ------------------------------------------------------------

ax1.set_xticks(
    x
)

ax1.set_xticklabels(
    metric_labels
)

ax1.set_ylim(
    -0.04,
    1.08
)

ax1.set_yticks(
    np.linspace(
        0,
        1,
        6
    )
)

# FINAL WORDING FIX:
# avoid implying that higher always means "better".
ax1.set_ylabel(
    "Normalized discovery characteristic"
)

ax1.set_title(
    "(a) Statistical discovery behaviour",
    loc="left",
    fontweight="bold",
    pad=14
)

ax1.grid(
    axis="y",
    linestyle=":",
    linewidth=0.6,
    alpha=0.38
)

ax1.set_axisbelow(True)

ax1.spines[
    ["top", "right"]
].set_visible(False)

# ------------------------------------------------------------
# FINAL SPACING FIX:
# move normalization explanation into its own row beneath title
# ------------------------------------------------------------

ax1.text(
    0.0,
    1.012,
    (
        "Per-metric min–max normalization; "
        "worst-slice AUPRC is inverted to represent failure severity."
    ),
    transform=ax1.transAxes,
    ha="left",
    va="bottom",
    fontsize=6.1,
    color="0.32"
)

# ------------------------------------------------------------
# FINAL LEGEND SPACING FIX
# ------------------------------------------------------------

ax1.legend(
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(
        0.50,
        -0.225
    ),
    ncol=2,
    columnspacing=1.4,
    handletextpad=0.5,
    fontsize=6.7
)

# ============================================================
# PANEL (b) — COMPLETE ASAP QUALIFICATION
# ============================================================

asap_plot = (
    asap
    .sort_values(
        "qualification_rate",
        ascending=True
    )
    .reset_index(drop=True)
)

y = np.arange(
    len(asap_plot)
)

bar_colors = [
    METHOD_COLORS[m]
    for m in asap_plot["method"]
]

bars = ax2.barh(
    y,
    asap_plot["qualification_rate"],
    height=0.56,
    color=bar_colors,
    edgecolor="black",
    linewidth=0.55
)

ax2.set_yticks(
    y
)

ax2.set_yticklabels(
    asap_plot["method"],
    fontsize=7
)

ax2.set_xlim(
    0,
    0.60
)

ax2.set_xticks(
    [
        0.0,
        0.1,
        0.2,
        0.3,
        0.4,
        0.5,
        0.6
    ]
)

ax2.set_xticklabels(
    [
        "0",
        "10",
        "20",
        "30",
        "40",
        "50",
        "60"
    ]
)

ax2.set_xlabel(
    "Complete ASAP qualification (%)"
)

ax2.set_title(
    "(b) Complete ASAP qualification",
    loc="left",
    fontweight="bold",
    pad=14
)

# ------------------------------------------------------------
# Exact qualification labels
# ------------------------------------------------------------

for bar, (_, row) in zip(
    bars,
    asap_plot.iterrows()
):

    rate = float(
        row["qualification_rate"]
    )

    qualified = int(
        row["qualified"]
    )

    reported = int(
        row["reported"]
    )

    label = (
        f"{qualified}/{reported} "
        f"({100 * rate:.0f}%)"
    )

    if rate == 0:

        ax2.text(
            0.012,
            bar.get_y()
            + bar.get_height() / 2,
            label,
            va="center",
            ha="left",
            fontsize=7,
            fontweight="bold"
        )

    else:

        ax2.text(
            rate + 0.015,
            bar.get_y()
            + bar.get_height() / 2,
            label,
            va="center",
            ha="left",
            fontsize=7,
            fontweight="bold"
        )

# ------------------------------------------------------------
# 50% reference guide
# ------------------------------------------------------------

ax2.axvline(
    0.50,
    linestyle="--",
    linewidth=0.85,
    color="0.45",
    zorder=0
)

ax2.grid(
    axis="x",
    linestyle=":",
    linewidth=0.55,
    alpha=0.35
)

ax2.set_axisbelow(True)

ax2.spines[
    ["top", "right"]
].set_visible(False)

# ============================================================
# 11. FIGURE-LEVEL NOTE
# ============================================================

fig.text(
    0.5,
    0.018,
    (
        "Panel (a) describes statistical discovery characteristics; "
        "Panel (b) reports survival under the common complete-ASAP "
        "qualification protocol."
    ),
    ha="center",
    va="bottom",
    fontsize=6.8
)

# ============================================================
# 12. FINAL LAYOUT
#
# More bottom space for Panel A legend.
# ============================================================

fig.subplots_adjust(
    left=0.095,
    right=0.98,
    top=0.885,
    bottom=0.275,
    wspace=0.39
)

# ============================================================
# 13. EXPORT
# ============================================================

fig.savefig(
    PDF_PATH,
    format="pdf",
    bbox_inches="tight",
    pad_inches=0.10
)

fig.savefig(
    SVG_PATH,
    format="svg",
    bbox_inches="tight",
    pad_inches=0.10
)

fig.savefig(
    PNG_PATH,
    format="png",
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.10
)

# ============================================================
# 14. VERIFY EXPORTS
# ============================================================

print("\n" + "=" * 78)
print("FIGURE 5 EXPORT VERIFICATION")
print("=" * 78)

for label, path in {
    "PDF": PDF_PATH,
    "SVG": SVG_PATH,
    "PNG": PNG_PATH,
    "RAW DATA": RAW_DATA_PATH,
    "NORMALIZED DATA": NORMALIZED_DATA_PATH,
    "ASAP DATA": ASAP_DATA_PATH
}.items():

    if not path.exists():

        raise IOError(
            f"{label} was not created: {path}"
        )

    size_kb = (
        path.stat().st_size
        / 1024
    )

    if size_kb <= 0:

        raise IOError(
            f"{label} appears empty: {path}"
        )

    print(
        f"{label}: {path.resolve()}"
    )

    print(
        f"     size = {size_kb:.1f} KB"
    )

plt.show()
plt.close(fig)

# ============================================================
# 15. REPORT EXACT DATA
# ============================================================

print("\n" + "=" * 78)
print("FIGURE 5 GENERATED SUCCESSFULLY")
print("=" * 78)

print("\nPanel (a) raw values:")

display(
    profile.style.format({
        "mean_harm": "{:.4f}",
        "max_harm": "{:.4f}",
        "worst_slice_auprc": "{:.4f}",
        "true_union_coverage": "{:.2%}"
    })
)

print("\nPanel (a) normalized values:")

display(
    profile_norm[
        [
            "method",
            "mean_harm_norm",
            "max_harm_norm",
            "worst_auprc_severity_norm",
            "coverage_norm"
        ]
    ]
    .style
    .format({
        "mean_harm_norm": "{:.3f}",
        "max_harm_norm": "{:.3f}",
        "worst_auprc_severity_norm": "{:.3f}",
        "coverage_norm": "{:.3f}"
    })
)

print("\nPanel (b) complete ASAP qualification:")

display(
    asap.style.format({
        "qualification_rate": "{:.1%}"
    })
)

print("\nInterpretation:")
print(
    "Panel (a) is a descriptive discovery profile, not an "
    "overall method-quality score."
)

print(
    "Panel (b) independently reports complete-ASAP qualification."
)

print("\nFigure directory:")
print(FIG_DIR.resolve())

print("=" * 78)

In [ ]:
# ============================================================
# DIAGNOSTIC — FIND THE CORRECT SOURCE FOR FIGURE 6 PANEL (b)
# ============================================================

from pathlib import Path
import pandas as pd

candidate_files = [
    "nb8_asap_constrained_weight_grid_rankings.csv",
    "nb8_asap_constrained_weight_grid_summary.csv",
    "nb8_asap_constrained_weight_grid_diagnostics.csv",
    "nb9_basic_rank_weight_justification_table.csv",
    "nb9_basic_rank_weight_sensitivity.csv"
]

print("=" * 90)
print("FIGURE 6 PANEL (b) — RANKER SENSITIVITY SOURCE DIAGNOSTIC")
print("=" * 90)

for file_name in candidate_files:

    path = Path(file_name)

    print("\n" + "-" * 90)
    print("FILE:", file_name)
    print("-" * 90)

    if not path.exists():
        print("NOT FOUND")
        continue

    df_test = pd.read_csv(path)

    print("Shape:", df_test.shape)
    print("\nColumns:")
    print(df_test.columns.tolist())

    print("\nFirst 5 rows:")
    display(df_test.head())

    print("\nPotential ranking / sensitivity columns:")

    keywords = [
        "scheme",
        "rank",
        "slice",
        "weight",
        "jaccard",
        "overlap",
        "spearman",
        "top5",
        "top7",
        "top10"
    ]

    candidate_cols = [
        c for c in df_test.columns
        if any(
            keyword in str(c).lower()
            for keyword in keywords
        )
    ]

    print(candidate_cols)

print("\n" + "=" * 90)
print("END OF DIAGNOSTIC")
print("=" * 90)

In [ ]:
# ============================================================
# FIGURE 6 — GOVERNANCE AND REPAIR-INFLUENCE ROBUSTNESS
# FINAL IEEE PUBLICATION VERSION — LAYOUT-FIXED
#
# Panel (a): One-at-a-time ASAP guardrail ablation
# Panel (b): Repair-influence weight sensitivity
#
# FINAL VISUAL FIXES:
#   1. Active-guardrail callouts moved into a dedicated
#      right-side annotation region.
#   2. Panel (a) legend moved upward.
#   3. Figure-level note moved lower with extra bottom margin.
#
# Data sources:
#   nb8_asap_guardrail_ablation_summary.csv
#   nb8_asap_constrained_weight_grid_summary.csv
#
# Outputs:
#   PDF -> vector
#   SVG -> vector
#   PNG -> 600 dpi
#   CSV -> exact source data
# ============================================================

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

warnings.filterwarnings("ignore")
plt.close("all")

# ============================================================
# 0. OUTPUT DIRECTORY
# ============================================================

FIG_DIR = Path.cwd() / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

PDF_PATH = FIG_DIR / "Fig6_Governance_and_Ranker_Robustness.pdf"
SVG_PATH = FIG_DIR / "Fig6_Governance_and_Ranker_Robustness.svg"
PNG_PATH = FIG_DIR / "Fig6_Governance_and_Ranker_Robustness.png"

ABLATION_EXPORT = FIG_DIR / "Fig6A_ASAP_Guardrail_Ablation.csv"
SENSITIVITY_EXPORT = FIG_DIR / "Fig6B_Ranker_Weight_Sensitivity.csv"
SENSITIVITY_SUMMARY_EXPORT = (
    FIG_DIR / "Fig6B_Ranker_Weight_Sensitivity_Summary.csv"
)

# ============================================================
# 1. IEEE-FRIENDLY TYPOGRAPHY
# ============================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "axes.linewidth": 0.8,
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# ============================================================
# 2. HELPERS
# ============================================================

def standardize_columns(df):

    df = df.copy()

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[\s\-/]+", "_", regex=True)
        .str.replace(r"[^a-z0-9_]", "", regex=True)
        .str.replace(r"_+", "_", regex=True)
        .str.strip("_")
    )

    return df


def clean_text(value):

    if pd.isna(value):
        return ""

    text = str(value).strip()

    if text.lower() in {
        "",
        "nan",
        "none",
        "[]"
    }:
        return ""

    return text


def canonical_ablation_name(configuration, removed_rule=""):

    config = (
        str(configuration)
        .strip()
        .lower()
        .replace("-", "_")
        .replace("/", "_")
        .replace(" ", "_")
    )

    removed = (
        str(removed_rule)
        .strip()
        .lower()
        .replace("-", "_")
        .replace("/", "_")
        .replace(" ", "_")
    )

    combined = f"{config} {removed}"

    if (
        "full_asap" in config
        or config == "full"
        or (
            "full" in config
            and removed in {
                "",
                "none",
                "nan"
            }
        )
    ):
        return "Full ASAP"

    if "plausib" in combined:
        return "Without plausibility"

    if (
        "replication" in combined
        or "replication_stability" in combined
        or (
            "stability" in combined
            and "artifact" not in combined
        )
    ):
        return "Without replication/stability"

    if "artifact" in combined:
        return "Without artifact veto"

    if "redundan" in combined:
        return "Without redundancy control"

    return str(configuration)


# ============================================================
# 3. PANEL (a) — GUARDRAIL ABLATION
# ============================================================

ABLATION_FILE = Path(
    "nb8_asap_guardrail_ablation_summary.csv"
)

if not ABLATION_FILE.exists():

    raise FileNotFoundError(
        "Could not find "
        "'nb8_asap_guardrail_ablation_summary.csv'."
    )

ablation_raw = pd.read_csv(
    ABLATION_FILE
)

ablation_raw = standardize_columns(
    ablation_raw
)

print("=" * 82)
print("FIGURE 6 — GOVERNANCE AND REPAIR-INFLUENCE ROBUSTNESS")
print("=" * 82)

print("\nPanel (a) source:")
print(ABLATION_FILE)

if "configuration" not in ablation_raw.columns:
    raise ValueError(
        "Expected column 'configuration' was not found."
    )

if "n_eligible_slices" not in ablation_raw.columns:
    raise ValueError(
        "Expected column 'n_eligible_slices' was not found."
    )

if "removed_rule" in ablation_raw.columns:

    removed_rule_series = (
        ablation_raw["removed_rule"]
    )

else:

    removed_rule_series = pd.Series(
        "",
        index=ablation_raw.index
    )

ablation = pd.DataFrame()

ablation["scenario"] = [
    canonical_ablation_name(
        cfg,
        rem
    )
    for cfg, rem in zip(
        ablation_raw["configuration"],
        removed_rule_series
    )
]

ablation["eligible_slices"] = pd.to_numeric(
    ablation_raw["n_eligible_slices"],
    errors="coerce"
)

if (
    "n_newly_admitted_vs_full_asap"
    in ablation_raw.columns
):

    ablation["n_newly_admitted"] = (
        pd.to_numeric(
            ablation_raw[
                "n_newly_admitted_vs_full_asap"
            ],
            errors="coerce"
        )
        .fillna(0)
        .astype(int)
    )

else:

    ablation["n_newly_admitted"] = 0


if (
    "newly_admitted_slices"
    in ablation_raw.columns
):

    ablation["newly_admitted_slices"] = (
        ablation_raw[
            "newly_admitted_slices"
        ]
        .apply(clean_text)
    )

else:

    ablation["newly_admitted_slices"] = ""


required_scenarios = [
    "Full ASAP",
    "Without plausibility",
    "Without replication/stability",
    "Without artifact veto",
    "Without redundancy control"
]

ablation = (
    ablation[
        ablation["scenario"].isin(
            required_scenarios
        )
    ]
    .drop_duplicates(
        subset="scenario",
        keep="first"
    )
    .set_index("scenario")
    .reindex(required_scenarios)
    .reset_index()
)

if ablation["eligible_slices"].isna().any():

    missing = (
        ablation.loc[
            ablation["eligible_slices"].isna(),
            "scenario"
        ]
        .tolist()
    )

    raise ValueError(
        "Could not recover these ablation scenarios:\n"
        f"{missing}"
    )

ablation["eligible_slices"] = (
    ablation["eligible_slices"]
    .astype(int)
)

expected_counts = {
    "Full ASAP": 11,
    "Without plausibility": 11,
    "Without replication/stability": 11,
    "Without artifact veto": 12,
    "Without redundancy control": 12
}

for _, row in ablation.iterrows():

    if (
        int(row["eligible_slices"])
        != expected_counts[row["scenario"]]
    ):

        raise ValueError(
            "Guardrail-ablation result does not match "
            "the finalized Figure 6 evidence."
        )

ablation.to_csv(
    ABLATION_EXPORT,
    index=False
)

# ============================================================
# 4. PANEL (b) — 39-SCHEME WEIGHT SENSITIVITY
# ============================================================

SENSITIVITY_FILE = Path(
    "nb8_asap_constrained_weight_grid_summary.csv"
)

if not SENSITIVITY_FILE.exists():

    raise FileNotFoundError(
        "Could not find "
        "'nb8_asap_constrained_weight_grid_summary.csv'."
    )

sensitivity_raw = pd.read_csv(
    SENSITIVITY_FILE
)

sensitivity_raw = standardize_columns(
    sensitivity_raw
)

required_sensitivity_columns = [
    "scheme",
    "top5_jaccard",
    "top7_jaccard",
    "top10_jaccard",
    "full_ranking_spearman"
]

missing_columns = [
    c
    for c in required_sensitivity_columns
    if c not in sensitivity_raw.columns
]

if missing_columns:

    raise ValueError(
        "Missing required sensitivity columns:\n"
        f"{missing_columns}"
    )

sensitivity = sensitivity_raw[
    required_sensitivity_columns
].copy()

sensitivity.columns = [
    "scheme",
    "Top-5 Jaccard",
    "Top-7 Jaccard",
    "Top-10 Jaccard",
    "Full-ranking Spearman"
]

for column in [
    "Top-5 Jaccard",
    "Top-7 Jaccard",
    "Top-10 Jaccard",
    "Full-ranking Spearman"
]:

    sensitivity[column] = pd.to_numeric(
        sensitivity[column],
        errors="coerce"
    )

sensitivity = sensitivity.dropna(
    subset=[
        "Top-5 Jaccard",
        "Top-7 Jaccard",
        "Top-10 Jaccard",
        "Full-ranking Spearman"
    ]
)

n_schemes = len(
    sensitivity
)

if n_schemes != 39:

    raise ValueError(
        f"Expected 39 weight schemes; found {n_schemes}."
    )

top10_values = (
    sensitivity["Top-10 Jaccard"]
    .to_numpy()
)

top10_all_fixed = np.allclose(
    top10_values,
    1.0,
    atol=1e-12
)

if not top10_all_fixed:

    raise ValueError(
        "Top-10 Jaccard is not fixed at 1.0 for all schemes."
    )

sensitivity.to_csv(
    SENSITIVITY_EXPORT,
    index=False
)

# ============================================================
# 5. DISTRIBUTION SUMMARY
# ============================================================

metric_order = [
    "Top-5 Jaccard",
    "Top-7 Jaccard",
    "Top-10 Jaccard",
    "Full-ranking Spearman"
]

sensitivity_summary = pd.DataFrame({
    "Metric": metric_order,

    "Min": [
        sensitivity[m].min()
        for m in metric_order
    ],

    "Q1": [
        sensitivity[m].quantile(0.25)
        for m in metric_order
    ],

    "Median": [
        sensitivity[m].median()
        for m in metric_order
    ],

    "Mean": [
        sensitivity[m].mean()
        for m in metric_order
    ],

    "Q3": [
        sensitivity[m].quantile(0.75)
        for m in metric_order
    ],

    "Max": [
        sensitivity[m].max()
        for m in metric_order
    ]
})

sensitivity_summary.to_csv(
    SENSITIVITY_SUMMARY_EXPORT,
    index=False
)

# ============================================================
# 6. CREATE FIGURE
# ============================================================

fig = plt.figure(
    figsize=(
        7.16,
        5.10
    )
)

gs = fig.add_gridspec(
    1,
    2,
    width_ratios=[
        1.08,
        1.35
    ],
    wspace=0.50
)

ax1 = fig.add_subplot(
    gs[0, 0]
)

ax2 = fig.add_subplot(
    gs[0, 1]
)

# ============================================================
# PANEL (a)
# ============================================================

ablation_plot = (
    ablation
    .set_index("scenario")
    .loc[
        required_scenarios[::-1]
    ]
    .reset_index()
)

y = np.arange(
    len(ablation_plot)
)

COLOR_REFERENCE = "#404040"
COLOR_NONBINDING = "#AAAAAA"
COLOR_ACTIVE = "#D55E00"

bar_colors = []

for scenario in ablation_plot["scenario"]:

    if scenario == "Full ASAP":

        bar_colors.append(
            COLOR_REFERENCE
        )

    elif scenario in {
        "Without artifact veto",
        "Without redundancy control"
    }:

        bar_colors.append(
            COLOR_ACTIVE
        )

    else:

        bar_colors.append(
            COLOR_NONBINDING
        )

bars = ax1.barh(
    y,
    ablation_plot["eligible_slices"],
    height=0.55,
    color=bar_colors,
    edgecolor="black",
    linewidth=0.55
)

display_labels = []

for scenario in ablation_plot["scenario"]:

    if scenario == "Without replication/stability":

        display_labels.append(
            "Without replication/\nstability"
        )

    elif scenario == "Without redundancy control":

        display_labels.append(
            "Without redundancy\ncontrol"
        )

    elif scenario == "Without artifact veto":

        display_labels.append(
            "Without artifact\nveto"
        )

    else:

        display_labels.append(
            scenario
        )

ax1.set_yticks(
    y
)

ax1.set_yticklabels(
    display_labels,
    fontsize=6.8
)

# ------------------------------------------------------------
# FINAL FIX:
# extend x-axis to create dedicated annotation space
# ------------------------------------------------------------

ax1.set_xlim(
    0,
    16.6
)

ax1.set_xticks(
    np.arange(
        0,
        15,
        2
    )
)

ax1.set_xlabel(
    "Eligible slices"
)

ax1.set_title(
    "(a) One-at-a-time ASAP guardrail ablation",
    loc="left",
    fontweight="bold",
    fontsize=8.8,
    pad=11
)

# Full ASAP reference
ax1.axvline(
    11,
    linestyle="--",
    linewidth=0.9,
    color="0.38",
    zorder=0
)

# ------------------------------------------------------------
# Bar values
# ------------------------------------------------------------

for bar, (_, row) in zip(
    bars,
    ablation_plot.iterrows()
):

    value = int(
        row["eligible_slices"]
    )

    ax1.text(
        value + 0.15,
        bar.get_y()
        + bar.get_height()/2,
        str(value),
        va="center",
        ha="left",
        fontsize=7,
        fontweight="bold"
    )

# ------------------------------------------------------------
# FINAL FIX:
# place active-guardrail annotations in dedicated x region
# ------------------------------------------------------------

for i, row in ablation_plot.iterrows():

    scenario = row["scenario"]

    if scenario == "Without artifact veto":

        ax1.annotate(
            "artifact-associated\ncandidate admitted",
            xy=(
                12,
                i
            ),
            xytext=(
                14.05,
                i + 0.30
            ),
            textcoords="data",
            ha="left",
            va="center",
            fontsize=6.0,
            color="#9A3412",
            arrowprops=dict(
                arrowstyle="-",
                color="#9A3412",
                linewidth=0.8
            )
        )

    elif scenario == "Without redundancy control":

        ax1.annotate(
            "redundant alias\nadmitted",
            xy=(
                12,
                i
            ),
            xytext=(
                14.05,
                i - 0.30
            ),
            textcoords="data",
            ha="left",
            va="center",
            fontsize=6.0,
            color="#9A3412",
            arrowprops=dict(
                arrowstyle="-",
                color="#9A3412",
                linewidth=0.8
            )
        )

ax1.grid(
    axis="x",
    linestyle=":",
    linewidth=0.55,
    alpha=0.35
)

ax1.set_axisbelow(True)

ax1.spines[
    ["top", "right"]
].set_visible(False)

# ------------------------------------------------------------
# FINAL FIX:
# legend moved upward; not near figure note
# ------------------------------------------------------------

ablation_legend = [
    Patch(
        facecolor=COLOR_REFERENCE,
        edgecolor="black",
        linewidth=0.4,
        label="Full ASAP"
    ),
    Patch(
        facecolor=COLOR_NONBINDING,
        edgecolor="black",
        linewidth=0.4,
        label="No eligibility change"
    ),
    Patch(
        facecolor=COLOR_ACTIVE,
        edgecolor="black",
        linewidth=0.4,
        label="Additional candidate admitted"
    )
]

ax1.legend(
    handles=ablation_legend,
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(
        0.50,
        -0.11
    ),
    ncol=1,
    fontsize=6.2,
    labelspacing=0.35
)

# ============================================================
# PANEL (b)
# ============================================================

distribution_data = [
    sensitivity[m]
    .to_numpy()
    for m in metric_order
]

positions = np.arange(
    1,
    len(metric_order) + 1
)

box = ax2.boxplot(
    distribution_data,
    positions=positions,
    widths=0.46,
    patch_artist=True,
    showfliers=False,
    medianprops=dict(
        color="black",
        linewidth=1.15
    ),
    boxprops=dict(
        linewidth=0.8
    ),
    whiskerprops=dict(
        linewidth=0.8
    ),
    capprops=dict(
        linewidth=0.8
    )
)

box_colors = [
    "#56B4E9",
    "#009E73",
    "#D55E00",
    "#CC79A7"
]

for patch, color in zip(
    box["boxes"],
    box_colors
):

    patch.set_facecolor(
        color
    )

    patch.set_alpha(
        0.52
    )

# Jittered observations
rng = np.random.default_rng(
    42
)

for pos, metric, color in zip(
    positions,
    metric_order,
    box_colors
):

    values = (
        sensitivity[metric]
        .to_numpy()
    )

    jitter = rng.normal(
        0,
        0.045,
        len(values)
    )

    ax2.scatter(
        np.full(
            len(values),
            pos
        )
        + jitter,
        values,
        s=13,
        color=color,
        edgecolor="black",
        linewidth=0.20,
        alpha=0.72,
        zorder=4
    )

# Perfect agreement reference
ax2.axhline(
    1.0,
    linestyle="--",
    linewidth=0.9,
    color="0.40",
    zorder=0
)

# Top-10 annotation
ax2.annotate(
    (
        "Invariant Top-10 membership\n"
        "Jaccard = 1.0 for all 39 schemes"
    ),
    xy=(
        3.0,
        1.0
    ),
    xytext=(
        3.16,
        0.915
    ),
    ha="left",
    va="top",
    fontsize=6.3,
    fontweight="bold",
    color="#9A3412",
    arrowprops=dict(
        arrowstyle="->",
        color="#9A3412",
        linewidth=0.9
    )
)

ax2.set_xticks(
    positions
)

ax2.set_xticklabels(
    [
        "Top-5\nJaccard",
        "Top-7\nJaccard",
        "Top-10\nJaccard",
        "Full-ranking\nSpearman"
    ],
    fontsize=6.8
)

all_values = np.concatenate(
    distribution_data
)

y_min = max(
    0,
    float(
        np.nanmin(
            all_values
        )
        - 0.06
    )
)

ax2.set_ylim(
    y_min,
    1.035
)

ax2.set_ylabel(
    "Agreement with reference weighting"
)

ax2.set_title(
    "(b) Repair-influence weight sensitivity",
    loc="left",
    fontweight="bold",
    fontsize=8.8,
    pad=11
)

ax2.grid(
    axis="y",
    linestyle=":",
    linewidth=0.55,
    alpha=0.35
)

ax2.set_axisbelow(True)

ax2.spines[
    ["top", "right"]
].set_visible(False)

# ============================================================
# 7. FIGURE-LEVEL NOTE
#
# FINAL FIX:
# pushed lower with dedicated bottom margin.
# ============================================================

fig.text(
    0.5,
    0.015,
    (
        "Guardrail ablation identifies active governance safeguards; "
        "weight sensitivity separates Top-K membership robustness "
        "from exact ranking-order stability."
    ),
    ha="center",
    va="bottom",
    fontsize=6.7
)

# ============================================================
# 8. FINAL LAYOUT
# ============================================================

fig.subplots_adjust(
    left=0.185,
    right=0.985,
    top=0.90,
    bottom=0.255,
    wspace=0.50
)

# ============================================================
# 9. EXPORT
# ============================================================

fig.savefig(
    PDF_PATH,
    format="pdf",
    bbox_inches="tight",
    pad_inches=0.10
)

fig.savefig(
    SVG_PATH,
    format="svg",
    bbox_inches="tight",
    pad_inches=0.10
)

fig.savefig(
    PNG_PATH,
    format="png",
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.10
)

# ============================================================
# 10. VERIFY OUTPUTS
# ============================================================

print("\n" + "=" * 82)
print("FIGURE 6 EXPORT VERIFICATION")
print("=" * 82)

for label, path in {

    "PDF":
        PDF_PATH,

    "SVG":
        SVG_PATH,

    "PNG":
        PNG_PATH,

    "ABLATION DATA":
        ABLATION_EXPORT,

    "SENSITIVITY DATA":
        SENSITIVITY_EXPORT,

    "SENSITIVITY SUMMARY":
        SENSITIVITY_SUMMARY_EXPORT

}.items():

    if not path.exists():

        raise IOError(
            f"{label} was not created:\n{path}"
        )

    size_kb = (
        path.stat().st_size
        / 1024
    )

    if size_kb <= 0:

        raise IOError(
            f"{label} appears empty:\n{path}"
        )

    print(
        f"{label}: {path.resolve()}"
    )

    print(
        f"     size = {size_kb:.1f} KB"
    )

plt.show()
plt.close(fig)

# ============================================================
# 11. NUMERICAL REPORT
# ============================================================

print("\n" + "=" * 82)
print("FIGURE 6 GENERATED SUCCESSFULLY")
print("=" * 82)

print("\nPanel (a) — Guardrail ablation:")

display(
    ablation[
        [
            "scenario",
            "eligible_slices",
            "n_newly_admitted",
            "newly_admitted_slices"
        ]
    ]
)

print(
    "\nPanel (b) — Weight-sensitivity summary:"
)

display(
    sensitivity_summary.style.format({
        "Min": "{:.3f}",
        "Q1": "{:.3f}",
        "Median": "{:.3f}",
        "Mean": "{:.3f}",
        "Q3": "{:.3f}",
        "Max": "{:.3f}"
    })
)

print(
    "\nWeight schemes evaluated:",
    n_schemes
)

print(
    "Top-10 Jaccard fixed at 1.0:",
    top10_all_fixed
)

print("\nFigure directory:")
print(FIG_DIR.resolve())

print("=" * 82)

In [ ]:
# ============================================================
# FIGURE 7 — DOWNSTREAM REPAIR EFFECT
# FINAL IEEE PUBLICATION VERSION — FIXED FOR ACTUAL BOOTSTRAP SCHEMA
#
# Panel (a):
#   Repair-slice AUPRC across:
#       - ERM
#       - BASiC P3
#       - Combined Targeted
#
#   Metrics:
#       - Mean repair-slice AUPRC
#       - Worst repair-slice AUPRC
#
# Panel (b):
#   Principal harmful slice:
#       abdominal_pain_and_hr_missing
#
#   Separate aligned sections:
#       - AUPRC with 95% bootstrap CI
#       - Brier score with 95% bootstrap CI
#
# Actual bootstrap schema supported:
#   arm
#   slice_name
#   auprc_mean
#   auprc_ci_low
#   auprc_ci_high
#   recall_mean
#   recall_ci_low
#   recall_ci_high
#   brier_mean
#   brier_ci_low
#   brier_ci_high
#   n_successful_bootstraps
#
# Outputs:
#   PDF -> vector
#   SVG -> vector
#   PNG -> 600 dpi
#   CSV -> exact figure source data
# ============================================================

import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

warnings.filterwarnings("ignore")
plt.close("all")

# ============================================================
# 0. OUTPUT DIRECTORY
# ============================================================

FIG_DIR = Path.cwd() / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

PDF_PATH = FIG_DIR / "Fig7_Downstream_Repair_Effect.pdf"
SVG_PATH = FIG_DIR / "Fig7_Downstream_Repair_Effect.svg"
PNG_PATH = FIG_DIR / "Fig7_Downstream_Repair_Effect.png"

PANEL_A_DATA_PATH = FIG_DIR / "Fig7A_Repair_Slice_Performance.csv"
PANEL_B_DATA_PATH = FIG_DIR / "Fig7B_Principal_Slice_Bootstrap.csv"

# ============================================================
# 1. IEEE-FRIENDLY TYPOGRAPHY
# ============================================================

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8,
    "axes.titlesize": 9,
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "axes.linewidth": 0.8,
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "none"
})

# ============================================================
# 2. HELPERS
# ============================================================

def standardize_columns(df):
    df = df.copy()

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[\s\-/]+", "_", regex=True)
        .str.replace(r"[^a-z0-9_]", "", regex=True)
        .str.replace(r"_+", "_", regex=True)
        .str.strip("_")
    )

    return df


def canonical_slice_name(value):
    text = str(value).strip().lower()

    text = re.sub(
        r"[\s\-/]+",
        "_",
        text
    )

    text = re.sub(
        r"_+",
        "_",
        text
    )

    return text.strip("_")


def find_col(
    df,
    exact=None,
    keyword_groups=None,
    required=True,
    description=""
):
    exact = exact or []

    for c in exact:
        if c in df.columns:
            return c

    if keyword_groups:

        for group in keyword_groups:

            matches = []

            for c in df.columns:
                text = str(c).lower()

                if all(
                    keyword.lower() in text
                    for keyword in group
                ):
                    matches.append(c)

            if len(matches) == 1:
                return matches[0]

    if required:
        raise ValueError(
            f"\nCould not identify '{description}'.\n"
            f"Available columns:\n{list(df.columns)}"
        )

    return None


def pretty_arm_name(value):
    text = str(value).strip().lower()

    if (
        "erm" in text
        or "unweighted" in text
    ):
        return "ERM"

    if (
        "revised_p3" in text
        or "p3_provenance_both" in text
        or text == "p3"
        or "basic_p3" in text
    ):
        return "BASiC P3"

    if (
        "combined_targeted" in text
        or "combined targeted" in text
    ):
        return "Combined Targeted"

    return str(value)


# ============================================================
# 3. LOAD GOVERNANCE-QUALIFIED REPAIR-SLICE REGISTRY
# ============================================================

registry_candidates = [
    "nb8_repair_eligible_combined_slices.csv",
    "nb9_repair_eligible_slice_results.csv",
    "nb8_top_ranked_repair_slices.csv"
]

registry_file = next(
    (
        f
        for f in registry_candidates
        if Path(f).exists()
    ),
    None
)

if registry_file is None:
    raise FileNotFoundError(
        "\nCould not locate a repair-eligible slice registry.\n"
        "Expected one of:\n  - "
        + "\n  - ".join(registry_candidates)
    )

registry_raw = standardize_columns(
    pd.read_csv(
        registry_file
    )
)

registry_slice_col = find_col(
    registry_raw,
    exact=[
        "slice_name",
        "slice",
        "slice_id",
        "name"
    ],
    keyword_groups=[
        ["slice", "name"]
    ],
    description="repair-eligible slice name"
)

repair_slice_keys = set(
    registry_raw[
        registry_slice_col
    ]
    .dropna()
    .astype(str)
    .apply(canonical_slice_name)
)

if len(repair_slice_keys) == 0:
    raise ValueError(
        "The repair-eligible slice registry is empty."
    )

print("=" * 82)
print("FIGURE 7 — DOWNSTREAM REPAIR EFFECT")
print("=" * 82)

print("\nRepair-slice registry:")
print(" ", registry_file)

print(
    "Governance-qualified repair slices:",
    len(repair_slice_keys)
)

# ============================================================
# 4. PANEL (a) — ARM-SPECIFIC SLICE RESULTS
# ============================================================

arm_files = {
    "ERM":
        "nb9_slice_results__ERM_unweighted.csv",

    "BASiC P3":
        "nb9_slice_results__revised_P3_provenance_both.csv",

    "Combined Targeted":
        "nb9_slice_results__combined_targeted.csv"
}

for arm, filename in arm_files.items():

    if not Path(filename).exists():
        raise FileNotFoundError(
            f"\nMissing required Figure 7 file:\n"
            f"{filename}\n"
            f"Needed for arm: {arm}"
        )

panel_a_rows = []

for arm, filename in arm_files.items():

    temp = standardize_columns(
        pd.read_csv(
            filename
        )
    )

    slice_col = find_col(
        temp,
        exact=[
            "slice_name",
            "slice",
            "slice_id",
            "name"
        ],
        keyword_groups=[
            ["slice", "name"]
        ],
        description=f"{arm} slice name"
    )

    auprc_col = find_col(
        temp,
        exact=[
            "auprc",
            "slice_auprc",
            "val_auprc"
        ],
        keyword_groups=[
            ["auprc"]
        ],
        description=f"{arm} slice AUPRC"
    )

    working = temp[
        [
            slice_col,
            auprc_col
        ]
    ].copy()

    working.columns = [
        "slice_name",
        "AUPRC"
    ]

    working["slice_key"] = (
        working["slice_name"]
        .apply(canonical_slice_name)
    )

    working["AUPRC"] = pd.to_numeric(
        working["AUPRC"],
        errors="coerce"
    )

    working = working.dropna(
        subset=[
            "slice_key",
            "AUPRC"
        ]
    )

    working = working[
        working["slice_key"].isin(
            repair_slice_keys
        )
    ].copy()

    working = working.drop_duplicates(
        subset="slice_key",
        keep="first"
    )

    if working.empty:
        raise ValueError(
            f"No governance-qualified repair slices found in {filename}."
        )

    panel_a_rows.append({
        "arm":
            arm,

        "n_repair_slices":
            len(working),

        "mean_repair_slice_AUPRC":
            working["AUPRC"].mean(),

        "worst_repair_slice_AUPRC":
            working["AUPRC"].min(),

        "mean_to_worst_gap":
            (
                working["AUPRC"].mean()
                - working["AUPRC"].min()
            )
    })

panel_a = pd.DataFrame(
    panel_a_rows
)

arm_order = [
    "ERM",
    "BASiC P3",
    "Combined Targeted"
]

panel_a = (
    panel_a
    .set_index("arm")
    .loc[arm_order]
    .reset_index()
)

panel_a.to_csv(
    PANEL_A_DATA_PATH,
    index=False
)

# ============================================================
# 5. PANEL (b) — LOAD ACTUAL BOOTSTRAP FILE
# ============================================================

BOOTSTRAP_FILE = Path(
    "nb9_revised_bootstrap_main_slice.csv"
)

if not BOOTSTRAP_FILE.exists():
    raise FileNotFoundError(
        "\nCould not find:\n"
        "nb9_revised_bootstrap_main_slice.csv"
    )

boot = standardize_columns(
    pd.read_csv(
        BOOTSTRAP_FILE
    )
)

print("\nPanel (b) bootstrap source:")
print(" ", BOOTSTRAP_FILE)

print("\nBootstrap columns:")
print(
    boot.columns.tolist()
)

# ============================================================
# 6. VALIDATE ACTUAL BOOTSTRAP SCHEMA
# ============================================================

required_boot_cols = [
    "arm",
    "slice_name",
    "auprc_mean",
    "auprc_ci_low",
    "auprc_ci_high",
    "brier_mean",
    "brier_ci_low",
    "brier_ci_high",
    "n_successful_bootstraps"
]

missing_boot_cols = [
    c
    for c in required_boot_cols
    if c not in boot.columns
]

if missing_boot_cols:

    raise ValueError(
        "\nBootstrap file is missing required columns:\n"
        f"{missing_boot_cols}\n\n"
        f"Available columns:\n{boot.columns.tolist()}"
    )

# ============================================================
# 7. FILTER TO PRINCIPAL HARMFUL SLICE
# ============================================================

principal_slice = (
    "abdominal_pain_and_hr_missing"
)

boot["slice_key"] = (
    boot["slice_name"]
    .apply(canonical_slice_name)
)

boot["arm_clean"] = (
    boot["arm"]
    .apply(pretty_arm_name)
)

boot_main = boot[
    (
        boot["slice_key"]
        == principal_slice
    )
    &
    (
        boot["arm_clean"].isin(
            [
                "ERM",
                "BASiC P3"
            ]
        )
    )
].copy()

if boot_main.empty:

    raise ValueError(
        "\nNo ERM/BASiC P3 bootstrap rows found for:\n"
        f"{principal_slice}\n\n"
        "Detected arms:\n"
        f"{boot[['arm', 'arm_clean']].drop_duplicates().to_dict('records')}"
    )

# ============================================================
# 8. BUILD PANEL-B TABLE DIRECTLY
# ============================================================

for c in [
    "auprc_mean",
    "auprc_ci_low",
    "auprc_ci_high",
    "brier_mean",
    "brier_ci_low",
    "brier_ci_high",
    "n_successful_bootstraps"
]:

    boot_main[c] = pd.to_numeric(
        boot_main[c],
        errors="coerce"
    )

boot_main = boot_main.dropna(
    subset=[
        "auprc_mean",
        "auprc_ci_low",
        "auprc_ci_high",
        "brier_mean",
        "brier_ci_low",
        "brier_ci_high"
    ]
)

boot_main = (
    boot_main
    .drop_duplicates(
        subset="arm_clean",
        keep="first"
    )
)

required_arms = {
    "ERM",
    "BASiC P3"
}

observed_arms = set(
    boot_main["arm_clean"]
)

missing_arms = (
    required_arms
    - observed_arms
)

if missing_arms:

    raise ValueError(
        "\nMissing principal-slice bootstrap arm(s):\n"
        f"{sorted(missing_arms)}\n\n"
        f"Observed: {sorted(observed_arms)}"
    )

panel_b_rows = []

for _, row in boot_main.iterrows():

    arm = row["arm_clean"]

    panel_b_rows.append({
        "arm":
            arm,
        "metric":
            "AUPRC",
        "estimate":
            row["auprc_mean"],
        "ci_lower":
            row["auprc_ci_low"],
        "ci_upper":
            row["auprc_ci_high"],
        "n_successful_bootstraps":
            int(
                row["n_successful_bootstraps"]
            )
    })

    panel_b_rows.append({
        "arm":
            arm,
        "metric":
            "Brier",
        "estimate":
            row["brier_mean"],
        "ci_lower":
            row["brier_ci_low"],
        "ci_upper":
            row["brier_ci_high"],
        "n_successful_bootstraps":
            int(
                row["n_successful_bootstraps"]
            )
    })

panel_b = pd.DataFrame(
    panel_b_rows
)

panel_b = (
    panel_b
    .set_index(
        [
            "metric",
            "arm"
        ]
    )
    .loc[
        [
            ("AUPRC", "ERM"),
            ("AUPRC", "BASiC P3"),
            ("Brier", "ERM"),
            ("Brier", "BASiC P3")
        ]
    ]
    .reset_index()
)

# ============================================================
# 9. VALIDATE CONFIDENCE INTERVALS
# ============================================================

bad_ci = panel_b[
    (
        panel_b["estimate"]
        < panel_b["ci_lower"]
    )
    |
    (
        panel_b["estimate"]
        > panel_b["ci_upper"]
    )
]

if not bad_ci.empty:

    raise ValueError(
        "At least one bootstrap point estimate lies "
        "outside its confidence interval."
    )

panel_b.to_csv(
    PANEL_B_DATA_PATH,
    index=False
)

# ============================================================
# 10. REPORT BOOTSTRAP COUNTS
# ============================================================

bootstrap_counts = (
    panel_b[
        "n_successful_bootstraps"
    ]
    .unique()
)

print(
    "\nSuccessful bootstrap counts:",
    bootstrap_counts.tolist()
)

if len(bootstrap_counts) == 1:

    bootstrap_note = (
        f"{int(bootstrap_counts[0]):,} successful resamples"
    )

else:

    bootstrap_note = (
        "bootstrap resamples as reported in Notebook 9"
    )

# ============================================================
# 11. CREATE FIGURE
# ============================================================

fig = plt.figure(
    figsize=(
        7.16,
        5.00
    )
)

outer = fig.add_gridspec(
    1,
    2,
    width_ratios=[
        1.25,
        1.0
    ],
    wspace=0.43
)

ax_a = fig.add_subplot(
    outer[0, 0]
)

right = outer[0, 1].subgridspec(
    2,
    1,
    hspace=0.48
)

ax_b1 = fig.add_subplot(
    right[0, 0]
)

ax_b2 = fig.add_subplot(
    right[1, 0]
)

# ============================================================
# 12. PANEL (a) — MEAN + WORST REPAIR-SLICE AUPRC
# ============================================================

METHOD_COLORS = {
    "ERM": "#7F7F7F",
    "BASiC P3": "#D55E00",
    "Combined Targeted": "#0072B2"
}

x = np.arange(
    len(arm_order)
)

width = 0.31

mean_values = (
    panel_a[
        "mean_repair_slice_AUPRC"
    ]
    .to_numpy()
)

worst_values = (
    panel_a[
        "worst_repair_slice_AUPRC"
    ]
    .to_numpy()
)

for i, arm in enumerate(
    arm_order
):

    color = METHOD_COLORS[
        arm
    ]

    ax_a.bar(
        x[i] - width/2,
        mean_values[i],
        width=width,
        color=color,
        edgecolor="black",
        linewidth=0.55,
        alpha=0.90,
        zorder=3
    )

    ax_a.bar(
        x[i] + width/2,
        worst_values[i],
        width=width,
        color=color,
        edgecolor="black",
        linewidth=0.65,
        alpha=0.38,
        hatch="//",
        zorder=3
    )

# Value labels
for i in range(
    len(arm_order)
):

    ax_a.text(
        x[i] - width/2,
        mean_values[i] + 0.008,
        f"{mean_values[i]:.3f}",
        ha="center",
        va="bottom",
        fontsize=6.4
    )

    ax_a.text(
        x[i] + width/2,
        worst_values[i] + 0.008,
        f"{worst_values[i]:.3f}",
        ha="center",
        va="bottom",
        fontsize=6.4
    )

ax_a.set_xticks(
    x
)

ax_a.set_xticklabels(
    [
        "ERM",
        "BASiC\nP3",
        "Combined\nTargeted"
    ]
)

all_a = np.concatenate(
    [
        mean_values,
        worst_values
    ]
)

a_min = max(
    0,
    float(
        np.nanmin(
            all_a
        )
        - 0.08
    )
)

a_max = min(
    1.0,
    float(
        np.nanmax(
            all_a
        )
        + 0.08
    )
)

ax_a.set_ylim(
    a_min,
    a_max
)

ax_a.set_ylabel(
    "Repair-slice AUPRC"
)

ax_a.set_title(
    "(a) Governance-qualified repair-slice performance",
    loc="left",
    fontweight="bold",
    fontsize=8.8,
    pad=11
)

ax_a.grid(
    axis="y",
    linestyle=":",
    linewidth=0.55,
    alpha=0.35
)

ax_a.set_axisbelow(True)

ax_a.spines[
    ["top", "right"]
].set_visible(False)

metric_handles = [
    Line2D(
        [0],
        [0],
        marker="s",
        linestyle="",
        markerfacecolor="0.35",
        markeredgecolor="black",
        markersize=7,
        label="Mean repair-slice AUPRC"
    ),

    Line2D(
        [0],
        [0],
        marker="s",
        linestyle="",
        markerfacecolor="0.80",
        markeredgecolor="black",
        markersize=7,
        label="Worst repair-slice AUPRC"
    )
]

ax_a.legend(
    handles=metric_handles,
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(
        0.50,
        -0.15
    ),
    ncol=1,
    fontsize=6.4
)

# ============================================================
# 13. PANEL (b1) — AUPRC
# ============================================================

PANEL_B_COLORS = {
    "ERM": "#7F7F7F",
    "BASiC P3": "#D55E00"
}

panel_b_auprc = (
    panel_b[
        panel_b["metric"] == "AUPRC"
    ]
    .set_index("arm")
    .loc[
        [
            "ERM",
            "BASiC P3"
        ]
    ]
    .reset_index()
)

pb_x = np.arange(
    2
)

for i, row in panel_b_auprc.iterrows():

    estimate = float(
        row["estimate"]
    )

    lower = float(
        row["ci_lower"]
    )

    upper = float(
        row["ci_upper"]
    )

    ax_b1.errorbar(
        i,
        estimate,
        yerr=np.array(
            [
                [estimate - lower],
                [upper - estimate]
            ]
        ),
        fmt="o",
        markersize=6.2,
        capsize=4,
        linewidth=1.2,
        color=PANEL_B_COLORS[
            row["arm"]
        ],
        markeredgecolor="black",
        markeredgewidth=0.55,
        zorder=4
    )

    ax_b1.text(
        i + 0.07,
        estimate,
        f"{estimate:.3f}",
        va="center",
        ha="left",
        fontsize=6.5,
        fontweight="bold"
    )

auprc_min = (
    panel_b_auprc[
        "ci_lower"
    ].min()
)

auprc_max = (
    panel_b_auprc[
        "ci_upper"
    ].max()
)

auprc_span = max(
    0.03,
    auprc_max - auprc_min
)

ax_b1.set_ylim(
    max(
        0,
        auprc_min
        - 0.20 * auprc_span
    ),
    min(
        1,
        auprc_max
        + 0.22 * auprc_span
    )
)

ax_b1.set_xticks(
    pb_x
)

ax_b1.set_xticklabels(
    [
        "ERM",
        "BASiC P3"
    ]
)

ax_b1.set_ylabel(
    "AUPRC"
)

ax_b1.set_title(
    "(b) Principal harmful slice: abdominal pain + HR missing",
    loc="left",
    fontweight="bold",
    fontsize=8.5,
    pad=10
)

ax_b1.text(
    0.0,
    1.015,
    f"95% bootstrap interval; {bootstrap_note}",
    transform=ax_b1.transAxes,
    ha="left",
    va="bottom",
    fontsize=6.1,
    color="0.30"
)

ax_b1.grid(
    axis="y",
    linestyle=":",
    linewidth=0.55,
    alpha=0.35
)

ax_b1.set_axisbelow(True)

ax_b1.spines[
    ["top", "right"]
].set_visible(False)

# ============================================================
# 14. PANEL (b2) — BRIER SCORE
# ============================================================

panel_b_brier = (
    panel_b[
        panel_b["metric"] == "Brier"
    ]
    .set_index("arm")
    .loc[
        [
            "ERM",
            "BASiC P3"
        ]
    ]
    .reset_index()
)

for i, row in panel_b_brier.iterrows():

    estimate = float(
        row["estimate"]
    )

    lower = float(
        row["ci_lower"]
    )

    upper = float(
        row["ci_upper"]
    )

    ax_b2.errorbar(
        i,
        estimate,
        yerr=np.array(
            [
                [estimate - lower],
                [upper - estimate]
            ]
        ),
        fmt="o",
        markersize=6.2,
        capsize=4,
        linewidth=1.2,
        color=PANEL_B_COLORS[
            row["arm"]
        ],
        markeredgecolor="black",
        markeredgewidth=0.55,
        zorder=4
    )

    ax_b2.text(
        i + 0.07,
        estimate,
        f"{estimate:.3f}",
        va="center",
        ha="left",
        fontsize=6.5,
        fontweight="bold"
    )

brier_min = (
    panel_b_brier[
        "ci_lower"
    ].min()
)

brier_max = (
    panel_b_brier[
        "ci_upper"
    ].max()
)

brier_span = max(
    0.01,
    brier_max - brier_min
)

ax_b2.set_ylim(
    max(
        0,
        brier_min
        - 0.25 * brier_span
    ),
    brier_max
    + 0.25 * brier_span
)

ax_b2.set_xticks(
    pb_x
)

ax_b2.set_xticklabels(
    [
        "ERM",
        "BASiC P3"
    ]
)

ax_b2.set_ylabel(
    "Brier score"
)

ax_b2.text(
    0.0,
    1.015,
    "Lower is better",
    transform=ax_b2.transAxes,
    ha="left",
    va="bottom",
    fontsize=6.1,
    color="0.30"
)

ax_b2.grid(
    axis="y",
    linestyle=":",
    linewidth=0.55,
    alpha=0.35
)

ax_b2.set_axisbelow(True)

ax_b2.spines[
    ["top", "right"]
].set_visible(False)

# ============================================================
# 15. FIGURE-LEVEL NOTE
# ============================================================

fig.text(
    0.5,
    0.018,
    (
        "Panel (a) summarizes downstream performance across governance-qualified "
        "repair slices; Panel (b) isolates the principal harmful slice and "
        "reports bootstrap uncertainty for discrimination and probabilistic error."
    ),
    ha="center",
    va="bottom",
    fontsize=6.7
)

# ============================================================
# 16. FINAL LAYOUT
# ============================================================

fig.subplots_adjust(
    left=0.09,
    right=0.985,
    top=0.90,
    bottom=0.205,
    wspace=0.43
)

# ============================================================
# 17. EXPORT
# ============================================================

fig.savefig(
    PDF_PATH,
    format="pdf",
    bbox_inches="tight",
    pad_inches=0.10
)

fig.savefig(
    SVG_PATH,
    format="svg",
    bbox_inches="tight",
    pad_inches=0.10
)

fig.savefig(
    PNG_PATH,
    format="png",
    dpi=600,
    bbox_inches="tight",
    pad_inches=0.10
)

# ============================================================
# 18. VERIFY OUTPUTS
# ============================================================

print("\n" + "=" * 82)
print("FIGURE 7 EXPORT VERIFICATION")
print("=" * 82)

for label, path in {

    "PDF":
        PDF_PATH,

    "SVG":
        SVG_PATH,

    "PNG":
        PNG_PATH,

    "PANEL A DATA":
        PANEL_A_DATA_PATH,

    "PANEL B DATA":
        PANEL_B_DATA_PATH

}.items():

    if not path.exists():
        raise IOError(
            f"{label} was not created:\n{path}"
        )

    size_kb = (
        path.stat().st_size
        / 1024
    )

    if size_kb <= 0:
        raise IOError(
            f"{label} appears empty:\n{path}"
        )

    print(
        f"{label}: {path.resolve()}"
    )

    print(
        f"     size = {size_kb:.1f} KB"
    )

plt.show()
plt.close(fig)

# ============================================================
# 19. FINAL NUMERICAL REPORT
# ============================================================

print("\n" + "=" * 82)
print("FIGURE 7 GENERATED SUCCESSFULLY")
print("=" * 82)

print("\nPanel (a):")

display(
    panel_a.style.format({
        "mean_repair_slice_AUPRC": "{:.4f}",
        "worst_repair_slice_AUPRC": "{:.4f}",
        "mean_to_worst_gap": "{:.4f}"
    })
)

print("\nPanel (b):")

display(
    panel_b.style.format({
        "estimate": "{:.4f}",
        "ci_lower": "{:.4f}",
        "ci_upper": "{:.4f}"
    })
)

print("\nBootstrap source:")
print(BOOTSTRAP_FILE)

print("\nBootstrap note:")
print(bootstrap_note)

print("\nInterpretation:")
print(
    "Panel (a) compares downstream performance across the "
    "same governance-qualified repair slices."
)

print(
    "Panel (b) evaluates the principal harmful slice using "
    "bootstrap uncertainty for AUPRC and Brier score."
)

print("\nFigure directory:")
print(FIG_DIR.resolve())

print("=" * 82)

Keep only final accepted slices

In [ ]:
accepted_registry = registry[
    registry["final_accept"].astype(str).str.lower().isin(["yes", "true", "1"])
].copy()

print("Accepted slices:", len(accepted_registry))
accepted_registry[["slice_name", "slice_type_manual", "val_support_pct", "priority_score"]]

Create a clean accepted slice summary table

In [ ]:
slice_summary = accepted_registry[[
    "slice_name",
    "slice_type_manual",
    "train_support_pct",
    "val_support_pct",
    "val_logloss_lift",
    "val_recall_gap",
    "val_fn_lift",
    "val_hcc_lift",
    "priority_score"
]].copy()

slice_summary = slice_summary.sort_values("priority_score", ascending=False).reset_index(drop=True)
slice_summary

Create a compact top-10 table for slides

In [ ]:
top_slice_table = slice_summary.head(10).copy()

top_slice_table = top_slice_table.rename(columns={
    "slice_name": "Slice",
    "slice_type_manual": "Type",
    "train_support_pct": "Train Support (%)",
    "val_support_pct": "Val Support (%)",
    "val_logloss_lift": "Val LogLoss Lift",
    "val_recall_gap": "Val Recall Gap",
    "val_fn_lift": "Val FN Lift",
    "val_hcc_lift": "Val HCC Lift",
    "priority_score": "Priority"
})

top_slice_table

Count slice types

In [ ]:
type_counts = accepted_registry["slice_type_manual"].value_counts()
type_counts

Plot accepted slice type distribution

In [ ]:
plt.figure(figsize=(7, 4))
type_counts.plot(kind="bar")
plt.title("Accepted Harm Slice Types")
plt.ylabel("Number of Slices")
plt.xlabel("Slice Type")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("fig1_accepted_slice_type_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

Sort arms in the intended order

In [ ]:
arm_order = ["raw_erm", "slice_reweight", "suspicious_downweight", "combined_repair"]

global_results["arm"] = pd.Categorical(global_results["arm"], categories=arm_order, ordered=True)
global_results = global_results.sort_values("arm").reset_index(drop=True)

global_results

Plot global AUROC and AUPRC

In [ ]:
x = np.arange(len(global_results))
width = 0.35

plt.figure(figsize=(8, 4.5))
plt.bar(x - width/2, global_results["AUROC"], width, label="AUROC")
plt.bar(x + width/2, global_results["AUPRC"], width, label="AUPRC")

plt.xticks(x, global_results["arm"], rotation=20, ha="right")
plt.ylabel("Score")
plt.title("Global Discrimination Across Repair Arms")
plt.ylim(0.55, 0.85)
plt.legend()
plt.tight_layout()
plt.savefig("fig2_global_discrimination_across_arms.png", dpi=300, bbox_inches="tight")
plt.show()

Plot global Recall and Precision at 0.5

In [ ]:
x = np.arange(len(global_results))
width = 0.35

plt.figure(figsize=(8, 4.5))
plt.bar(x - width/2, global_results["Recall@0.5"], width, label="Recall@0.5")
plt.bar(x + width/2, global_results["Precision@0.5"], width, label="Precision@0.5")

plt.xticks(x, global_results["arm"], rotation=20, ha="right")
plt.ylabel("Score")
plt.title("Global Operating Point Across Repair Arms (Threshold = 0.5)")
plt.ylim(0.45, 0.70)
plt.legend()
plt.tight_layout()
plt.savefig("fig3_global_operating_point_across_arms.png", dpi=300, bbox_inches="tight")
plt.show()

Prepare compact robustness table

In [ ]:
robustness_plot_df = worst_slice_results.copy()

robustness_plot_df["arm"] = pd.Categorical(robustness_plot_df["arm"], categories=arm_order, ordered=True)
robustness_plot_df = robustness_plot_df.sort_values("arm").reset_index(drop=True)

robustness_plot_df[[
    "arm",
    "mean_slice_AUPRC",
    "worst_slice_AUPRC",
    "mean_slice_Recall@0.5",
    "worst_slice_Recall@0.5"
]]

Plot mean vs worst-slice AUPRC

In [ ]:
x = np.arange(len(robustness_plot_df))
width = 0.35

plt.figure(figsize=(8.5, 4.8))
plt.bar(x - width/2, robustness_plot_df["mean_slice_AUPRC"], width, label="Mean Slice AUPRC")
plt.bar(x + width/2, robustness_plot_df["worst_slice_AUPRC"], width, label="Worst Slice AUPRC")

plt.xticks(x, robustness_plot_df["arm"], rotation=20, ha="right")
plt.ylabel("AUPRC")
plt.title("Mean vs Worst Accepted-Slice AUPRC Across Repair Arms")
plt.ylim(0.0, 0.75)
plt.legend()
plt.tight_layout()
plt.savefig("fig4_mean_vs_worst_slice_auprc.png", dpi=300, bbox_inches="tight")
plt.show()

Plot mean vs worst-slice Recall@0.5

In [ ]:
x = np.arange(len(robustness_plot_df))
width = 0.35

plt.figure(figsize=(8.5, 4.8))
plt.bar(x - width/2, robustness_plot_df["mean_slice_Recall@0.5"], width, label="Mean Slice Recall@0.5")
plt.bar(x + width/2, robustness_plot_df["worst_slice_Recall@0.5"], width, label="Worst Slice Recall@0.5")

plt.xticks(x, robustness_plot_df["arm"], rotation=20, ha="right")
plt.ylabel("Recall")
plt.title("Mean vs Worst Accepted-Slice Recall Across Repair Arms")
plt.ylim(0.0, 0.65)
plt.legend()
plt.tight_layout()
plt.savefig("fig5_mean_vs_worst_slice_recall.png", dpi=300, bbox_inches="tight")
plt.show()

Plot average-to-worst AUPRC gap

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.bar(
    robustness_plot_df["arm"],
    robustness_plot_df["avg_to_worst_AUPRC_gap"]
)
plt.ylabel("Gap")
plt.title("Average-to-Worst Accepted-Slice AUPRC Gap")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("fig6_avg_to_worst_auprc_gap.png", dpi=300, bbox_inches="tight")
plt.show()

Plot average-to-worst Recall gap

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.bar(
    robustness_plot_df["arm"],
    robustness_plot_df["avg_to_worst_Recall_gap"]
)
plt.ylabel("Gap")
plt.title("Average-to-Worst Accepted-Slice Recall Gap")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("fig7_avg_to_worst_recall_gap.png", dpi=300, bbox_inches="tight")
plt.show()

Prepare delta table merged with slice type

In [ ]:
delta_plot_df = delta_combined.merge(
    accepted_registry[["slice_name", "slice_type_manual"]],
    on="slice_name",
    how="left"
)

delta_plot_df = delta_plot_df.sort_values("delta_Recall_vs_raw", ascending=False).reset_index(drop=True)
delta_plot_df[["slice_name", "slice_type_manual", "delta_AUPRC_vs_raw", "delta_Recall_vs_raw"]]

Plot horizontal bar chart for Recall delta

In [ ]:
plt.figure(figsize=(9, 6))
plt.barh(delta_plot_df["slice_name"], delta_plot_df["delta_Recall_vs_raw"])
plt.axvline(0, linestyle="--")
plt.xlabel("Δ Recall@0.5 vs Raw ERM")
plt.ylabel("Accepted Slice")
plt.title("Combined Repair: Slice-Level Recall Change vs Raw ERM")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("fig8_combined_repair_delta_recall.png", dpi=300, bbox_inches="tight")
plt.show()

Plot horizontal bar chart for AUPRC delta

In [ ]:
plt.figure(figsize=(9, 6))
plt.barh(delta_plot_df["slice_name"], delta_plot_df["delta_AUPRC_vs_raw"])
plt.axvline(0, linestyle="--")
plt.xlabel("Δ AUPRC vs Raw ERM")
plt.ylabel("Accepted Slice")
plt.title("Combined Repair: Slice-Level AUPRC Change vs Raw ERM")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("fig9_combined_repair_delta_auprc.png", dpi=300, bbox_inches="tight")
plt.show()

Extract the hardest slice (lowest raw AUPRC)

In [ ]:
hardest_slice = delta_plot_df.sort_values("raw_AUPRC", ascending=True).iloc[0]
hardest_slice

Create a spotlight table for the hardest slice

In [ ]:
hardest_slice_summary = pd.DataFrame({
    "Metric": ["Raw AUPRC", "Combined AUPRC", "Raw Recall@0.5", "Combined Recall@0.5"],
    "Value": [
        hardest_slice["raw_AUPRC"],
        hardest_slice["AUPRC"],
        hardest_slice["raw_Recall@0.5"],
        hardest_slice["Recall@0.5"]
    ]
})

hardest_slice_name = hardest_slice["slice_name"]
print("Hardest slice:", hardest_slice_name)
hardest_slice_summary

Plot spotlight figure for the hardest slice

In [ ]:
plt.figure(figsize=(7, 4.5))
plt.bar(hardest_slice_summary["Metric"], hardest_slice_summary["Value"])
plt.ylabel("Value")
plt.title(f"Hardest Hidden Slice Spotlight: {hardest_slice_name}")
plt.xticks(rotation=20, ha="right")
plt.ylim(0, max(0.2, hardest_slice_summary["Value"].max() + 0.05))
plt.tight_layout()
plt.savefig("fig10_hardest_slice_spotlight.png", dpi=300, bbox_inches="tight")
plt.show()

Build the integrated summary table

In [ ]:
summary_table = global_results.merge(
    worst_slice_results[[
        "arm",
        "mean_slice_AUPRC",
        "worst_slice_AUPRC",
        "mean_slice_Recall@0.5",
        "worst_slice_Recall@0.5",
        "avg_to_worst_AUPRC_gap",
        "avg_to_worst_Recall_gap"
    ]],
    on="arm",
    how="left"
)

summary_table = summary_table[[
    "arm",
    "AUROC", "AUPRC", "Brier",
    "Recall@0.5", "Precision@0.5",
    "mean_slice_AUPRC", "worst_slice_AUPRC",
    "mean_slice_Recall@0.5", "worst_slice_Recall@0.5",
    "avg_to_worst_AUPRC_gap", "avg_to_worst_Recall_gap"
]]

summary_table

Save the integrated summary table

In [ ]:
summary_table.to_csv("nb7_integrated_summary_table.csv", index=False)
print("Saved nb7_integrated_summary_table.csv")

Create a simple interpretation text block (Optional)

In [ ]:
best_mean_recall_arm = robustness_plot_df.loc[
    robustness_plot_df["mean_slice_Recall@0.5"].idxmax(), "arm"
]

best_worst_auprc_arm = robustness_plot_df.loc[
    robustness_plot_df["worst_slice_AUPRC"].idxmax(), "arm"
]

print("WEEKLY PRESENTATION TALKING POINTS")
print("=" * 60)
print(f"1. Final accepted harmful slice registry contains {len(accepted_registry)} slices.")
print(f"2. Best mean slice recall arm: {best_mean_recall_arm}")
print(f"3. Best worst-slice AUPRC arm: {best_worst_auprc_arm}")
print(f"4. Hardest slice identified: {hardest_slice_name}")
print("5. Global AUROC/AUPRC remain stable across all repair arms.")
print("6. Worst-slice recall remains unresolved, motivating targeted next-stage repair.")